<a href="https://colab.research.google.com/github/AntonDozhdikov/Demography_migration/blob/main/82_region_MADDPG_MATD3_SR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RF Demography & Migration MARL Pipeline — full rebuild

Полная версия ноутбука, сохранённая в исходном объёме и структуре, но переподключённая к `Itogovaia-baza-dannykh.csv` по всем субъектам РФ.

Что изменено принципиально:
- источник данных: только `Itogovaia-baza-dannykh.csv`;
- исключены строки федеральных округов;
- удалены автозаполнения отсутствующих state features нулями;
- удалены фиктивные переменные по умолчанию;
- оставлена полная структура world model / env / MADDPG / MATD3 / Evo-SR / evaluation / export.


# 🇷🇺 Моделирование управления демографией и миграцией по регионам РФ
## Multi-Agent RL + Evolutionary World-Model Pipeline  
`MADDPG` · `MATD3` · `Evo-MATD3-SR` (Self-Referential Evolutionary)

---
**Данные:** Итоговая база данных 1991–2024, все субъекты РФ  
**Горизонт прогноза:** 2025–2050 (26 лет)  
**Запуск:** Runtime → Run all  
> ⚠️ Загрузите `Itogovaia-baza-dannykh.csv` в `/content/` перед запуском


In [1]:
# ── [1] Установка зависимостей ────────────────────────────────────────────────
import subprocess, sys

PKGS = ["statsmodels", "tqdm"]
for p in PKGS:
    try:
        __import__(p)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])

print("✅ Зависимости готовы")


✅ Зависимости готовы


In [2]:
# ── [2] Импорты и настройка воспроизводимости ─────────────────────────────────
import os, copy, time, json, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error

from statsmodels.tsa.holtwinters import ExponentialSmoothing

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm, trange

# Воспроизводимость
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Устройство (автоматически T4 GPU в Colab)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Устройство: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


In [3]:
# ── [3] Конфигурация эксперимента ─────────────────────────────────────────────
from dataclasses import dataclass, field

@dataclass
class Config:
    # ── Пути ─────────────────────────────────────────────────────────────────
    DATA_PATH:  str = "Itogovaia-baza-dannykh.csv"
    OUTPUT_DIR: str = "output"
    SEED:       int = 42

    # ── Временные разбивки ────────────────────────────────────────────────────
    YEAR_TRAIN_END:   int = 2012
    YEAR_VAL_END:     int = 2018
    FORECAST_HORIZON: int = 26          # 2025-2050

    # ── Признаки состояния (14 числовых + 9 ФО = 23) ─────────────────────────
    STATE_FEATURES: List[str] = field(default_factory=lambda: [
        "population", "life_expectancy", "tfr", "urban_ratio",
        "unemployment", "real_wage_growth", "poverty_rate",
        "migration_balance_rate", "grp_per_capita", "grp_growth",
        "hospital_beds", "housing_per_capita", "internet_access", "cpi",
    ])
    STATE_DIM:  int = 23
    ACTION_DIM: int = 9

    # ── Названия 9 управляющих политик ────────────────────────────────────────
    ACTION_NAMES: List[str] = field(default_factory=lambda: [
        "pronatal_support",           # Пронаталистские меры
        "maternal_child_health",      # Охрана материнства и детства
        "mortality_reduction",        # Снижение смертности / профилактика
        "internal_migrant_attraction",# Привлечение внутренних мигрантов
        "youth_retention",            # Удержание молодёжи
        "labour_market_adaptation",   # Адаптация рынка труда
        "housing_infrastructure",     # Жильё и инфраструктура
        "education_human_capital",    # Образование и человеческий капитал
        "crisis_resilience_buffer",   # Резерв устойчивости к кризисам
    ])
    BUDGET_LIMIT: float = 3.5          # максимальная сумма всех действий агента

    # ── Веса компонент reward ─────────────────────────────────────────────────
    REWARD_WEIGHTS: Dict[str, float] = field(default_factory=lambda: {
        "population":      0.25,
        "life_expectancy": 0.20,
        "fertility":       0.15,
        "migration":       0.15,
        "poverty":         0.15,
        "unemployment":    0.05,
        "budget":          0.05,
    })

    # ── World Model (Ensemble GRU) ─────────────────────────────────────────────
    WM_ENSEMBLE_SIZE: int   = 3
    WM_HIDDEN_DIM:    int   = 128
    WM_N_LAYERS:      int   = 2
    WM_DROPOUT:       float = 0.15
    WM_LR:            float = 3e-4
    WM_EPOCHS:        int   = 1000
    WM_BATCH_SIZE:    int   = 256
    WM_PATIENCE:      int   = 100

    # ── Кризисы ───────────────────────────────────────────────────────────────
    CRISIS_INJECT_PROB: float = 0.15    # вероятность умеренного кризиса
    TAIL_RISK_PROB:     float = 0.04    # вероятность хвостового кризиса

    # ── MADDPG ────────────────────────────────────────────────────────────────
    MADDPG_EPISODES:     int   = 1000
    MADDPG_EPISODE_LEN:  int   = 15
    MADDPG_BATCH_SIZE:   int   = 256
    MADDPG_BUFFER_SIZE:  int   = 50_000
    MADDPG_LR_ACTOR:     float = 1e-4
    MADDPG_LR_CRITIC:    float = 3e-4
    MADDPG_GAMMA:        float = 0.97
    MADDPG_TAU:          float = 0.005
    MADDPG_WARMUP:       int   = 300
    MADDPG_HIDDEN:       int   = 256
    MADDPG_LOG_INTERVAL: int   = 50

    # ── MATD3 ─────────────────────────────────────────────────────────────────
    MATD3_EPISODES:      int   = 1000
    MATD3_LR_ACTOR:      float = 1e-4
    MATD3_LR_CRITIC:     float = 3e-4
    MATD3_GAMMA:         float = 0.97
    MATD3_TAU:           float = 0.005
    MATD3_POLICY_DELAY:  int   = 2
    MATD3_TARGET_NOISE:  float = 0.10
    MATD3_NOISE_CLIP:    float = 0.20
    MATD3_HIDDEN:        int   = 256
    MATD3_LOG_INTERVAL:  int   = 50

    # ── Evo-MATD3-SR ──────────────────────────────────────────────────────────
    EVO_EPISODES: int = 1000
    EVO_POP_SIZE: int = 8
    EVO_ELITE_FRAC: float = 0.5
    EVO_MUTATION_STD: float = 0.025
    EVO_CROSSOVER_PROB: float = 0.60
    EVO_META_INTERVAL: int = 10
    EVO_PLATEAU_PATIENCE: int = 20
    EVO_ARCHIVE_SIZE: int = 8
    EVO_LOG_INTERVAL: int = 50

    EVO_FAST_HORIZON: int = 5
    EVO_REFINE_HORIZON: int = 10
    EVO_REFINE_TOPK: int = 4
    EVO_REFINE_EPISODES: int = 3
    EVO_FAST_WEIGHT: float = 0.45
    EVO_REFINE_WEIGHT: float = 0.55
    EVO_FIT_MIN_EPISODES: int = 2

    EVO_MUTATE_FIRST_SCALE: float = 0.35
    EVO_MUTATE_MIDDLE_SCALE: float = 1.00
    EVO_MUTATE_LAST_SCALE: float = 2.20

    EVO_ARCHIVE_CROSSOVER_INTERVAL: int = 2
    EVO_ARCHIVE_PARENT_PROB: float = 0.45

    EVO_BASE_INJECT_TAU: float = 0.35
    EVO_BASE_INJECT_TOPK: int = 2
    EVO_POP_RESET_INTERVAL: int = 30
    EVO_SIGMA_MAX: float = 0.35
    EVO_MUT_STD_MAX: float = 0.06

    RL_PATIENCE: int = 100

    FIGURE_STYLE: str = "seaborn-v0_8-whitegrid"
    FIGURE_DPI: int = 120


CFG = Config()

# Создаём директории вывода
for d in ["output/checkpoints", "output/figures", "output/tables", "output/logs"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ Конфигурация загружена")
print(f"  State dim: {CFG.STATE_DIM}  |  Action dim: {CFG.ACTION_DIM}")
print(f"  Эпизоды: MADDPG={CFG.MADDPG_EPISODES} MATD3={CFG.MATD3_EPISODES} Evo={CFG.EVO_EPISODES}")


✅ Конфигурация загружена
  State dim: 23  |  Action dim: 9
  Эпизоды: MADDPG=1000 MATD3=1000 Evo=1000


In [4]:
# ── [4] Утилиты: Timer, EarlyStopping, ExperimentLogger, ArtifactManager ──────

class Timer:
    """Контекстный менеджер для измерения времени выполнения блоков."""
    def __init__(self, name: str = ""):
        self.name = name

    def __enter__(self):
        self._t0 = time.time()
        return self

    def __exit__(self, *_):
        print(f"  ⏱  {self.name}: {time.time() - self._t0:.1f}s")


class EarlyStopping:
    """
    Ранняя остановка с восстановлением лучших весов.
    mode='min' для loss  |  mode='max' для reward/score
    """
    def __init__(self, patience=20, mode="min", delta=1e-5, restore_best=True):
        self.patience = patience
        self.mode = mode
        self.delta = delta
        self.restore_best = restore_best
        self.best_score = np.inf if mode == "min" else -np.inf
        self.counter = 0
        self._best_state = None

    def _is_better(self, score):
        if self.mode == "min":
            return score < self.best_score - self.delta
        return score > self.best_score + self.delta

    def step(self, score, model: Optional[nn.Module] = None) -> bool:
        """Возвращает True → нужно остановить обучение."""
        if self._is_better(score):
            self.best_score = score
            self.counter = 0
            if model is not None and self.restore_best:
                self._best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1

        if self.counter >= self.patience:
            if model is not None and self._best_state is not None and self.restore_best:
                model.load_state_dict(self._best_state)
            return True
        return False


class ExperimentLogger:
    """Логирование метрик RL-обучения по эпизодам."""
    def __init__(self, name: str):
        self.name = name
        self.records: List[Dict] = []

    def _to_serializable(self, obj):
        if isinstance(obj, dict):
            return {str(k): self._to_serializable(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [self._to_serializable(v) for v in obj]
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        return obj

    def log(self, episode: int, metrics: Dict[str, float]):
        safe_metrics = {k: self._to_serializable(v) for k, v in metrics.items()}
        self.records.append({
            "episode": int(episode),
            **safe_metrics
        })

    def save(self):
        path = f"output/logs/{self.name}_log.json"
        safe_records = self._to_serializable(self.records)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(safe_records, f, indent=2, ensure_ascii=False)

    def to_df(self):
        return pd.DataFrame(self.records)


class ArtifactManager:
    """Единая точка сохранения таблиц и фигур."""
    def save_table(self, df: pd.DataFrame, name: str) -> str:
        path = f"output/tables/{name}.csv"
        df.to_csv(path, index=True)
        return path

    def save_figure(self, fig, name: str, dpi: int = 120) -> str:
        path = f"output/figures/{name}.png"
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return path


AM = ArtifactManager()
print("✅ Утилиты инициализированы")

✅ Утилиты инициализированы


In [5]:
# =============================================================================
# FULL RF DATA LOADING FROM REAL PANEL DATASET
# =============================================================================
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path('Itogovaia-baza-dannykh.csv')
assert DATA_PATH.exists(), f'Not found: {DATA_PATH}'

raw = pd.read_csv(DATA_PATH)
raw = raw[raw['Тип'].eq('Субъект РФ')].copy()
raw['Год'] = raw['Год'].astype(int)
raw = raw.sort_values(['Регион', 'Год']).reset_index(drop=True)

RU_COLS = {
    'region': 'Регион',
    'year': 'Год',
    'population': 'Численность постоянного населения на 1 января',
    'life_expectancy': 'Ожидаемая продолжительность жизни при рождении',
    'tfr': 'Суммарный коэффициент рождаемости (всего)',
    'tfr_1': 'Суммарный коэффициент рождаемости первых детей',
    'tfr_2': 'Суммарный коэффициент рождаемости вторых детей',
    'tfr_3p': 'Суммарный коэффициент рождаемости третьих и последующих детей',
    'urban_ratio': 'Доля городского населения в общей численности населения на 1 января',
    'unemployment': 'Уровень безработицы (по методологии МОТ)',
    'real_wage_growth': 'Реальная начисленная заработная плата, % к предыдущему году',
    'poverty_rate': 'Уровень бедности, %',
    'poor_share': 'Доля населения с доходами ниже ПМ, %',
    'housing_input': 'Введено в действие жилой площади, тыс. кв. м',
    'broadband': 'Доля домохозяйств с широкополосным доступом к Интернету, %',
    'housing_area_pc': 'Общая площадь жилых помещений на 1 жителя, кв. м',
    'families_need_housing': 'Число семей, нуждающихся в жилом помещении, тыс.',
    'arrived': 'Прибыло',
    'departed': 'Выбыло',
    'migration_balance': 'Миграционное сальдо',
    'grp_pc': 'ВРП на душу населения, руб.',
    'grp_growth': 'Индекс физического объема ВРП на душу населения, %',
    'retail_growth': 'Индекс физического объема оборота розничной торговли, %',
    'services_growth': 'Индекс физического объема платных услуг населению, %',
    'fixed_basket_growth': 'Индекс изменения стоимости фиксированного набора, %',
    'cpi': 'Индекс потребительских цен (декабрь к декабрю), %',
    'housing_price_index': 'Индекс цен на первичном рынке жилья, %',
    'fixed_basket_cost': 'Стоимость фиксированного набора потребительских товаров и услуг, ₽',
    'preschool_coverage': 'Валовой коэффициент охвата дошкольным образованием, %',
    'preschool_slots': 'Обеспеченность местами в ДОУ (на 1000 детей)',
    'ambulatory_capacity': 'Мощность амбулаторно-поликлинических организаций, посещений в смену',
    'beds_per_10k': 'Обеспеченность больничными койками (на 10 тыс. населения)',
    'disabled_k': 'Численность инвалидов, тыс. человек',
    'doctors': 'Численность врачей, чел.',
    'beds': 'Число больничных коек, ед.',
    'visits': 'Число посещений врачей, всего',
    'abortions': 'Число абортов, всего',
    'covid': 'Число зарегистрированных заболеваний: COVID-19',
    'disease_total': 'Число зарегистрированных заболеваний: Всего заболеваний',
    'blood': 'Число зарегистрированных заболеваний: Болезни крови и кроветворных органов',
    'resp': 'Число зарегистрированных заболеваний: Болезни органов дыхания',
    'digest': 'Число зарегистрированных заболеваний: Болезни органов пищеварения',
    'circul': 'Число зарегистрированных заболеваний: Болезни системы кровообращения',
    'infect': 'Число зарегистрированных заболеваний: Некоторые инфекционные и паразитарные болезни',
    'neo': 'Число зарегистрированных заболеваний: Новообразования',
}

present = {k: v for k, v in RU_COLS.items() if v in raw.columns}
df = raw[[v for v in present.values()]].copy().rename(columns={v: k for k, v in present.items()})

for c in df.columns:
    if c not in ['region', 'year']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

if {'migration_balance', 'population'}.issubset(df.columns):
    df['migration_balance_rate'] = df['migration_balance'] / df['population'] * 10000.0

if {'arrived', 'departed', 'population'}.issubset(df.columns):
    df['migration_turnover_rate'] = (df['arrived'] + df['departed']) / df['population'] * 10000.0

if {'abortions', 'population'}.issubset(df.columns):
    df['abortions_per_1000_pop'] = df['abortions'] / df['population'] * 1000.0

if {'visits', 'population'}.issubset(df.columns):
    df['doctor_visits_per_capita'] = df['visits'] / df['population']

if {'beds', 'population'}.issubset(df.columns):
    df['beds_per_capita'] = df['beds'] / df['population'] * 10000.0

if {'doctors', 'population'}.issubset(df.columns):
    df['doctors_per_capita'] = df['doctors'] / df['population'] * 10000.0

STATE_FEATURES = [c for c in df.columns if c not in ['region', 'year']]

parts = []
for reg, g in df.groupby('region'):
    g = g.sort_values('year').copy()
    for c in STATE_FEATURES:
        g[c] = g[c].interpolate(limit_area='inside')
    parts.append(g)

panel_df = pd.concat(parts, ignore_index=True)

valid_regions = panel_df.groupby('region')['year'].nunique()
valid_regions = valid_regions[valid_regions >= 10].index.tolist()
panel_df = panel_df[panel_df['region'].isin(valid_regions)].copy()

FEAT_COLS = STATE_FEATURES[:]

print('panel_df:', panel_df.shape)
print('regions:', panel_df['region'].nunique())
print('state_dim:', len(FEAT_COLS))
print('sample columns:', FEAT_COLS[:10])

panel_df: (3108, 51)
regions: 89
state_dim: 49
sample columns: ['population', 'life_expectancy', 'tfr', 'tfr_1', 'tfr_2', 'tfr_3p', 'urban_ratio', 'unemployment', 'real_wage_growth', 'poverty_rate']


In [6]:
# ── [6] Инженерия признаков, нормализация, split ──────────────────────────────

with Timer("Preprocessing"):

    # Производные признаки
    if "migration_balance" in df.columns and "population" in df.columns:
        df["migration_balance_rate"] = (
            df["migration_balance"] / df["population"].replace(0, np.nan) * 1000
        )

    if "grp_per_capita" in df.columns:
        df["grp_per_capita"] = np.log1p(df["grp_per_capita"].clip(lower=0))

    # Берём только реально существующие признаки из конфига
    FEAT_COLS = [f for f in CFG.STATE_FEATURES if f in df.columns]

    if len(FEAT_COLS) == 0:
        raise ValueError("Нет ни одного признака из CFG.STATE_FEATURES в датафрейме df")

    print(f"  Найдено признаков состояния: {len(FEAT_COLS)}")

    # Фильтрация регионов по покрытию (≥35%)
    coverage = (
        df.groupby("region")[FEAT_COLS]
        .apply(lambda g: g.notna().mean().mean())
        .sort_values(ascending=False)
    )
    GOOD_REGIONS = coverage[coverage >= 0.35].index.tolist()
    df = df[df["region"].isin(GOOD_REGIONS)].copy()
    print(f"  Регионов с покрытием ≥35%: {len(GOOD_REGIONS)}")

    # Интерполяция пропусков внутри региона
    def fill_ts(grp):
        grp = grp.sort_values("year").copy()
        grp[FEAT_COLS] = (
            grp[FEAT_COLS]
            .interpolate(method="linear", limit_direction="both")
            .ffill()
            .bfill()
        )
        return grp

    df = df.groupby("region", group_keys=False).apply(fill_ts)

    # Медианная импутация остатков
    imp = SimpleImputer(strategy="median")
    df[FEAT_COLS] = imp.fit_transform(df[FEAT_COLS])

    # RobustScaler (устойчив к выбросам)
    scaler = RobustScaler()
    df_norm = df.copy()
    df_norm[FEAT_COLS] = scaler.fit_transform(df[FEAT_COLS])

    # One-hot для федеральных округов, если столбец существует
    FD_COLS = []
    if "federal_district" in df_norm.columns:
        fd_ohe = pd.get_dummies(df_norm["federal_district"], prefix="fd").astype(float)

        if "FD_NAMES" in globals():
            for fd in FD_NAMES:
                col = f"fd_{fd}"
                if col not in fd_ohe.columns:
                    fd_ohe[col] = 0.0
            FD_COLS = [f"fd_{fd}" for fd in FD_NAMES]
        else:
            FD_COLS = sorted(fd_ohe.columns.tolist())

        df_norm = pd.concat(
            [df_norm.reset_index(drop=True), fd_ohe[FD_COLS].reset_index(drop=True)],
            axis=1
        )
    else:
        print("  Столбец federal_district отсутствует, one-hot ФО пропущен")

    ALL_STATE_COLS = FEAT_COLS + FD_COLS
    CFG.STATE_DIM = len(ALL_STATE_COLS)
    print(f"  Dim state: {CFG.STATE_DIM} (feat={len(FEAT_COLS)}, fd={len(FD_COLS)})")

    # Временные разбивки
    df_train = df_norm[df_norm["year"] <= CFG.YEAR_TRAIN_END].copy()
    df_val = df_norm[
        (df_norm["year"] > CFG.YEAR_TRAIN_END) &
        (df_norm["year"] <= CFG.YEAR_VAL_END)
    ].copy()
    df_test = df_norm[df_norm["year"] > CFG.YEAR_VAL_END].copy()
    print(f"  Train {len(df_train)} | Val {len(df_val)} | Test {len(df_test)}")

    # Индекс регионов
    regions = sorted(df_norm["region"].unique())
    N_REGIONS = len(regions)
    REG_IDX = {r: i for i, r in enumerate(regions)}
    print(f"  Регионов в модели: {N_REGIONS}")

    # Матрица смежности (ФО-граф), если есть federal_district
    adj_matrix = np.zeros((N_REGIONS, N_REGIONS), dtype=np.float32)

    if "federal_district" in df_norm.columns:
        fd_of = df_norm.groupby("region")["federal_district"].first().to_dict()
        for r1 in regions:
            for r2 in regions:
                if r1 != r2 and fd_of.get(r1) == fd_of.get(r2):
                    adj_matrix[REG_IDX[r1], REG_IDX[r2]] = 1.0
    else:
        print("  Столбец federal_district отсутствует, граф ФО будет пустым")

    row_sum = adj_matrix.sum(axis=1, keepdims=True).clip(min=1)
    adj_norm_mat = adj_matrix / row_sum
    ADJ_TENSOR = torch.tensor(adj_norm_mat, dtype=torch.float32, device=DEVICE)
    print(f"  Граф: {N_REGIONS}x{N_REGIONS}, рёбер: {int(adj_matrix.sum())}")

AM.save_table(
    coverage.reset_index().rename(columns={0: "coverage"}),
    "02_region_coverage"
)
print("\n✅ Предобработка завершена")

  Найдено признаков состояния: 10
  Регионов с покрытием ≥35%: 82
  Столбец federal_district отсутствует, one-hot ФО пропущен
  Dim state: 10 (feat=10, fd=0)
  Train 1886 | Val 492 | Test 574
  Регионов в модели: 82
  Столбец federal_district отсутствует, граф ФО будет пустым
  Граф: 82x82, рёбер: 0
  ⏱  Preprocessing: 0.6s

✅ Предобработка завершена


In [7]:
# ── [7] Кластеризация регионов + индекс уязвимости ────────────────────────────

with Timer("Кластеризация"):
    reg_profile = df_norm.groupby("region")[FEAT_COLS].mean().fillna(0)

    pca2 = PCA(n_components=2, random_state=CFG.SEED)
    reg_pca = pca2.fit_transform(reg_profile)

    N_CLUSTERS = 6
    km = KMeans(n_clusters=N_CLUSTERS, n_init=30, random_state=CFG.SEED)
    cluster_labels = km.fit_predict(reg_profile)
    CLUSTER_MAP = dict(zip(reg_profile.index, cluster_labels))

    # Индекс уязвимости: бедность + безработица - ВРП - миграция
    vuln_parts = []
    for feat, w in [("poverty_rate",0.35),("unemployment",0.30),
                    ("migration_balance_rate",-0.20),("grp_per_capita",-0.15)]:
        if feat in reg_profile.columns:
            vuln_parts.append(reg_profile[feat]*w)
    vuln_raw = sum(vuln_parts) if vuln_parts else pd.Series(0.5, index=reg_profile.index)
    v_min,v_max = vuln_raw.min(), vuln_raw.max()
    vuln_norm = (vuln_raw-v_min)/(v_max-v_min+1e-8)
    VULN_MAP  = vuln_norm.to_dict()
    print(f"  Кластеров: {N_CLUSTERS} | PCA: {pca2.explained_variance_ratio_.sum()*100:.1f}%")

    # ── Визуализация ──────────────────────────────────────────────────────────
    plt.style.use(CFG.FIGURE_STYLE)
    pal6 = sns.color_palette("tab10", N_CLUSTERS)
    fig, axes = plt.subplots(1,2,figsize=(14,6))
    fig.suptitle("Кластеризация регионов РФ по демографо-экономическому профилю",
                 fontsize=12, fontweight="bold")

    ax = axes[0]
    for cl in range(N_CLUSTERS):
        mask = cluster_labels==cl
        ax.scatter(reg_pca[mask,0],reg_pca[mask,1],c=[pal6[cl]],
                   label=f"К{cl+1}",s=55,alpha=0.8)
    ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title("PCA + K-Means")
    ax.legend(fontsize=8,ncol=2)

    ax2 = axes[1]
    top25 = vuln_norm.sort_values(ascending=False).head(25)
    colors_b = [pal6[CLUSTER_MAP.get(r,0)] for r in top25.index]
    ax2.barh(range(len(top25)), top25.values, color=colors_b, alpha=0.85)
    ax2.set_yticks(range(len(top25)))
    ax2.set_yticklabels(top25.index, fontsize=7)
    ax2.set_xlabel("Индекс уязвимости")
    ax2.set_title("ТОП-25 уязвимых регионов")
    ax2.invert_yaxis()
    plt.tight_layout()
    AM.save_figure(fig, "01_region_clusters")
    print("  → output/figures/01_region_clusters.png")

cluster_df = reg_profile.copy()
cluster_df["cluster"]       = [CLUSTER_MAP.get(r,-1) for r in cluster_df.index]
cluster_df["vulnerability"] = [VULN_MAP.get(r,0)     for r in cluster_df.index]
AM.save_table(cluster_df, "03_region_clusters")
print("✅ Кластеризация завершена")


  Кластеров: 6 | PCA: 75.9%
  → output/figures/01_region_clusters.png
  ⏱  Кластеризация: 0.8s
✅ Кластеризация завершена


In [8]:
# ── [8] Инерционный прогноз до 2050 (без управления) ──────────────────────────

def baseline_forecast(series: pd.Series, horizon: int) -> np.ndarray:
    """Holt-Winters ETS; при отказе → линейный тренд."""
    vals = series.dropna().values
    if len(vals) < 4:
        return np.full(horizon, vals[-1] if len(vals)>0 else 0.0)
    try:
        m = ExponentialSmoothing(vals, trend="add", seasonal=None,
                                 initialization_method="estimated")
        return m.fit(optimized=True, disp=False).forecast(horizon)
    except Exception:
        x = np.arange(len(vals))
        slope, intercept, *_ = stats.linregress(x, vals)
        return np.array([intercept+slope*(len(vals)+i) for i in range(1,horizon+1)])


with Timer("Baseline forecast 2025-2050"):
    FORECAST_YEARS = list(range(2025, 2025+CFG.FORECAST_HORIZON))
    baseline_records = []

    for region in tqdm(regions, desc="Baseline forecast", leave=False):
        rdf = df[df["region"]==region].sort_values("year")
        row = {"region": region}
        for feat in CFG.STATE_FEATURES:
            if feat in rdf.columns:
                preds = baseline_forecast(rdf.set_index("year")[feat], CFG.FORECAST_HORIZON)
            else:
                preds = np.zeros(CFG.FORECAST_HORIZON)
            for yr,val in zip(FORECAST_YEARS, preds):
                row[f"{feat}_{yr}"] = val
        baseline_records.append(row)

    # Long-формат для анализа
    bl_long = []
    for rec in baseline_records:
        r = rec["region"]
        for yr in FORECAST_YEARS:
            row2 = {"region":r,"year":yr}
            for feat in CFG.STATE_FEATURES:
                row2[feat] = rec.get(f"{feat}_{yr}", np.nan)
            bl_long.append(row2)
    df_baseline = pd.DataFrame(bl_long)

print(f"Инерционный прогноз: {len(df_baseline)} строк")
AM.save_table(df_baseline.head(2000), "04_baseline_forecast_sample")

# ── Визуализация baseline ─────────────────────────────────────────────────────
SHOWCASE = [r for r in [
    "г. Москва","Курская область","Республика Дагестан",
    "Сахалинская область","Новосибирская область","Краснодарский край"
] if r in regions][:6]
PLOT_FEATS = ["population","life_expectancy","tfr","migration_balance_rate"]

plt.style.use(CFG.FIGURE_STYLE)
fig, axes = plt.subplots(len(PLOT_FEATS),1,figsize=(14,10))
fig.suptitle("Инерционный прогноз ключевых показателей (без управления)",
             fontsize=12,fontweight="bold")
pal6s = sns.color_palette("husl", len(SHOWCASE))

for ai,feat in enumerate(PLOT_FEATS):
    ax = axes[ai]
    for ri,region in enumerate(SHOWCASE):
        rh = df[df["region"]==region].sort_values("year")
        if feat in rh.columns:
            ax.plot(rh["year"], rh[feat], color=pal6s[ri], lw=2.0,
                    label=region if ai==0 else None)
        rb = df_baseline[df_baseline["region"]==region]
        if feat in rb.columns:
            ax.plot(rb["year"], rb[feat], color=pal6s[ri], lw=1.2, ls="--", alpha=0.7)
    ax.axvline(2024, color="gray", ls=":", lw=1.2, alpha=0.8)
    ax.set_ylabel(feat.replace("_"," "), fontsize=8)
    ax.tick_params(labelsize=8)
    ax.grid(True,alpha=0.4)

axes[0].legend(fontsize=7, loc="upper right", ncol=2)
axes[-1].set_xlabel("Год")
plt.tight_layout()
AM.save_figure(fig,"02_baseline_forecasts")
print("  → output/figures/02_baseline_forecasts.png")
print("✅ Baseline forecast готов")


  ⏱  Baseline forecast 2025-2050: 1.2s
Инерционный прогноз: 2132 строк
  → output/figures/02_baseline_forecasts.png
✅ Baseline forecast готов


In [9]:
# ── [9] Архитектура ансамблевой GRU World Model ────────────────────────────────

class RegionTransitionNet(nn.Module):
    """
    GRU-сеть для предсказания следующего состояния одного региона.
    Вход:  state_i || action_i || neighbor_agg || crisis_ctx
    Выход: delta_state (среднее) + log_std (неопределённость)
    """
    def __init__(self, state_dim, action_dim, neigh_dim=None,
                 crisis_dim=4, hidden=128, n_layers=2, dropout=0.15):
        super().__init__()
        if neigh_dim is None: neigh_dim = state_dim
        input_dim = state_dim + action_dim + neigh_dim + crisis_dim
        self.gru = nn.GRU(input_dim, hidden, num_layers=n_layers,
                          batch_first=True,
                          dropout=dropout if n_layers>1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc_mean = nn.Sequential(
            nn.Linear(hidden,hidden), nn.LayerNorm(hidden), nn.SiLU(),
            nn.Linear(hidden,state_dim)
        )
        self.fc_std = nn.Linear(hidden, state_dim)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, s, a, neigh, crisis, hid=None):
        x = torch.cat([s,a,neigh,crisis], dim=-1).unsqueeze(1)   # [B,1,D]
        out, h = self.gru(x, hid)
        out = self.drop(out.squeeze(1))                           # [B,hidden]
        delta = self.fc_mean(out)
        log_std = self.fc_std(out).clamp(-4, 2)
        return delta, log_std, h


class EnsembleWorldModel(nn.Module):
    """
    Ансамбль K независимых GRU-моделей.
    Uncertainty = std по предсказаниям членов ансамбля (ensemble disagreement).
    """
    def __init__(self, state_dim, action_dim, ensemble_size=5, **kw):
        super().__init__()
        self.models = nn.ModuleList([
            RegionTransitionNet(state_dim, action_dim, **kw)
            for _ in range(ensemble_size)
        ])
        self.K = ensemble_size

    def forward(self, s, a, neigh, crisis):
        preds = [m(s,a,neigh,crisis)[0] for m in self.models]
        preds_t = torch.stack(preds, dim=0)         # [K,B,S]
        return preds_t.mean(0), preds_t.std(0)       # mean, uncertainty

    def single_forward(self, k, s, a, neigh, crisis, hid=None):
        return self.models[k](s,a,neigh,crisis,hid)


S_DIM = CFG.STATE_DIM
A_DIM = CFG.ACTION_DIM

world_model = EnsembleWorldModel(
    state_dim     = S_DIM,
    action_dim    = A_DIM,
    neigh_dim     = S_DIM,
    crisis_dim    = 4,
    ensemble_size = CFG.WM_ENSEMBLE_SIZE,
    hidden        = CFG.WM_HIDDEN_DIM,
    n_layers      = CFG.WM_N_LAYERS,
    dropout       = CFG.WM_DROPOUT,
).to(DEVICE)

total_p = sum(p.numel() for p in world_model.parameters())
print(f"EnsembleWorldModel: {total_p:,} параметров "
      f"({CFG.WM_ENSEMBLE_SIZE} моделей × {total_p//CFG.WM_ENSEMBLE_SIZE:,})")
print("✅ World Model архитектура создана")


EnsembleWorldModel: 543,036 параметров (3 моделей × 181,012)
✅ World Model архитектура создана


In [10]:
# ── [10] Обучение World Model ──────────────────────────────────────────────────

# Кризисные метки по историческим годам
CRISIS_YEAR_CTX = {}  # scenario layer should be injected explicitly during experiments

def build_wm_dataset(df_split, adj_mat, reg_idx, state_cols, action_dim, regions_list):
    """Строит (S_t, A_t, N_t, C_t, S_{t+1}) из исторических данных."""
    records = []
    years_s = sorted(df_split["year"].unique())
    for yi in range(len(years_s)-1):
        yr, yr1 = years_s[yi], years_s[yi+1]
        dfy  = df_split[df_split["year"]==yr].set_index("region")
        dfy1 = df_split[df_split["year"]==yr1].set_index("region")
        c_ctx = torch.tensor(CRISIS_YEAR_CTX.get(yr,[0,0,0,0]), dtype=torch.float32)
        for region in regions_list:
            if region not in dfy.index or region not in dfy1.index: continue
            ri = reg_idx.get(region,0)
            s_t  = torch.tensor(dfy.loc[region,state_cols].values.astype(float),
                                 dtype=torch.float32)
            s_t1 = torch.tensor(dfy1.loc[region,state_cols].values.astype(float),
                                 dtype=torch.float32)
            # Агрегат соседей: взвешенное среднее по adj
            n_s = torch.zeros(len(state_cols), dtype=torch.float32)
            for r2 in regions_list:
                w = adj_mat[ri, reg_idx.get(r2,0)]
                if w>0 and r2 in dfy.index:
                    n_s += w * torch.tensor(
                        dfy.loc[r2,state_cols].values.astype(float), dtype=torch.float32)
            a_t = torch.zeros(action_dim, dtype=torch.float32)
            s_t  = torch.nan_to_num(s_t)
            s_t1 = torch.nan_to_num(s_t1)
            n_s  = torch.nan_to_num(n_s)
            records.append((s_t, a_t, n_s, c_ctx, s_t1))
    if not records:
        z = lambda d: torch.zeros(1,d)
        return z(S_DIM),z(A_DIM),z(S_DIM),z(4),z(S_DIM)
    S,A,N,C,S1 = zip(*records)
    return (torch.stack(S), torch.stack(A), torch.stack(N),
            torch.stack([c.expand(4) for c in C]), torch.stack(S1))


print("Сборка датасетов для World Model...")
with Timer("Build WM datasets"):
    tr_S,tr_A,tr_N,tr_C,tr_S1 = build_wm_dataset(
        df_train,adj_norm_mat,REG_IDX,ALL_STATE_COLS,A_DIM,regions)
    va_S,va_A,va_N,va_C,va_S1 = build_wm_dataset(
        df_val,  adj_norm_mat,REG_IDX,ALL_STATE_COLS,A_DIM,regions)
print(f"  Train WM: {tr_S.shape[0]} | Val WM: {va_S.shape[0]} переходов")


def train_world_model(wm, tr_data, va_data, cfg):
    tr_S,tr_A,tr_N,tr_C,tr_S1 = [t.to(DEVICE) for t in tr_data]
    va_S,va_A,va_N,va_C,va_S1 = [t.to(DEVICE) for t in va_data]
    N_tr,B = tr_S.shape[0], cfg.WM_BATCH_SIZE
    hist = {"train":[],"val":[]}

    for k,model_k in enumerate(wm.models):
        opt = Adam(model_k.parameters(), lr=cfg.WM_LR, weight_decay=1e-5)
        sched = CosineAnnealingLR(opt, T_max=cfg.WM_EPOCHS, eta_min=1e-6)
        es_wm = EarlyStopping(patience=cfg.WM_PATIENCE, mode="min", restore_best=True)

        for ep in range(cfg.WM_EPOCHS):
            model_k.train()
            idx_list = torch.randperm(N_tr, device=DEVICE)
            ep_loss, nb = 0.0, 0
            for s in range(0,N_tr,B):
                idx = idx_list[s:s+B]
                if len(idx)<2: continue
                d_pred,ls,_ = model_k(tr_S[idx],tr_A[idx],tr_N[idx],tr_C[idx])
                d_true = tr_S1[idx] - tr_S[idx]
                var  = torch.exp(2*ls)+1e-8
                loss = ((d_true-d_pred)**2/(2*var)+ls).mean()
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model_k.parameters(),1.0)
                opt.step(); sched.step()
                ep_loss+=loss.item(); nb+=1
            tr_l = ep_loss/max(nb,1)

            model_k.eval()
            with torch.no_grad():
                dp,ls2,_ = model_k(va_S,va_A,va_N,va_C)
                dt = va_S1-va_S
                var2 = torch.exp(2*ls2)+1e-8
                va_l = ((dt-dp)**2/(2*var2)+ls2).mean().item()

            if k==0:
                hist["train"].append(tr_l); hist["val"].append(va_l)

            if es_wm.step(va_l, model_k):
                if k==0: print(f"  WM[{k}] early stop ep={ep+1} val={va_l:.4f}")
                break

        if k==0: print(f"  WM[{k}] best_val={es_wm.best_score:.4f}")
    return hist


with Timer("Обучение World Model"):
    wm_hist = train_world_model(
        world_model,(tr_S,tr_A,tr_N,tr_C,tr_S1),(va_S,va_A,va_N,va_C,va_S1),CFG)

torch.save(world_model.state_dict(),"output/checkpoints/world_model.pt")

fig,ax = plt.subplots(figsize=(8,4))
ax.plot(wm_hist["train"],label="Train NLL",color="steelblue")
ax.plot(wm_hist["val"],  label="Val NLL",  color="coral")
ax.set_xlabel("Эпоха"); ax.set_ylabel("NLL Loss")
ax.set_title("World Model: кривые обучения (ансамбль, модель #0)")
ax.legend(); ax.grid(True,alpha=0.4)
AM.save_figure(fig,"03_wm_learning_curve")
print("  → output/figures/03_wm_learning_curve.png")
print("✅ World Model обучена и сохранена")


Сборка датасетов для World Model...
  ⏱  Build WM datasets: 2.1s
  Train WM: 1804 | Val WM: 410 переходов
  WM[0] early stop ep=103 val=379.9943
  WM[0] best_val=0.7270
  ⏱  Обучение World Model: 19.5s
  → output/figures/03_wm_learning_curve.png
✅ World Model обучена и сохранена


In [11]:
# ── [11] Валидация World Model ────────────────────────────────────────────────

world_model.eval()
with torch.no_grad():
    va_S_d,va_A_d,va_N_d,va_C_d = [t.to(DEVICE) for t in (va_S,va_A,va_N,va_C)]
    m_pred, uncert = world_model(va_S_d,va_A_d,va_N_d,va_C_d)
    next_pred = (va_S_d + m_pred).cpu().numpy()
    next_true = va_S1.numpy()
    unc_np    = uncert.cpu().numpy()

mae_vals, unc_vals = {}, {}
n_feats = len(FEAT_COLS)
for fi,feat in enumerate(FEAT_COLS):
    if fi<next_pred.shape[1]:
        mae_vals[feat] = mean_absolute_error(next_true[:,fi],next_pred[:,fi])
        unc_vals[feat] = unc_np[:,fi].mean()

wm_mae_df = pd.DataFrame({
    "feature":list(mae_vals.keys()),
    "MAE":list(mae_vals.values()),
    "avg_uncertainty":list(unc_vals.values())
}).round(4)
print("World Model MAE (val set, normalized units):")
print(wm_mae_df.to_string(index=False))
AM.save_table(wm_mae_df,"05_world_model_mae")

# Scatter pred vs true
feat_list = list(mae_vals.keys())[:8]
fig,axes = plt.subplots(2,4,figsize=(14,7))
fig.suptitle("World Model: предсказание vs реальность (val set)",fontsize=12,fontweight="bold")
for i,(feat,ax) in enumerate(zip(feat_list,axes.flat)):
    fi = FEAT_COLS.index(feat)
    ax.scatter(next_true[:,fi],next_pred[:,fi],alpha=0.3,s=10,c="steelblue")
    lims = [min(next_true[:,fi].min(),next_pred[:,fi].min()),
            max(next_true[:,fi].max(),next_pred[:,fi].max())]
    ax.plot(lims,lims,"r--",lw=1.2)
    ax.set_title(feat.replace("_","\n"),fontsize=7)
    ax.set_xlabel("True",fontsize=7); ax.set_ylabel("Pred",fontsize=7)
    ax.text(0.05,0.92,f"MAE={mae_vals[feat]:.3f}",transform=ax.transAxes,
            fontsize=7,color="crimson")
    ax.tick_params(labelsize=6); ax.grid(True,alpha=0.3)
plt.tight_layout()
AM.save_figure(fig,"04_wm_validation")
print("  → output/figures/04_wm_validation.png")
print("✅ World Model валидация завершена")


World Model MAE (val set, normalized units):
               feature    MAE  avg_uncertainty
            population 0.0806           0.1964
       life_expectancy 0.1347           0.2291
                   tfr 0.1713           0.1398
           urban_ratio 0.0862           0.1719
          unemployment 0.0950           0.1247
      real_wage_growth 1.8460           0.1504
          poverty_rate 0.0598           0.2787
migration_balance_rate 0.3354           0.1706
            grp_growth 0.4339           0.2262
                   cpi 2.4210           0.1422
  → output/figures/04_wm_validation.png
✅ World Model валидация завершена


In [12]:
# ── [12] MARL Среда: RussiaRegionsEnv ─────────────────────────────────────────

# Кризисные сценарии (из crisis-3.txt + дополнения)
CRISIS_SCENARIOS = [
    {"id":1, "name":"COVID-19",       "b":-0.15,"d":0.25,"m":-0.6, "e":-0.2},
    {"id":2, "name":"Geopolitical",   "b":-0.20,"d":0.15,"m":-0.8, "e":-0.3},
    {"id":3, "name":"Ecological",     "b":-0.30,"d":0.40,"m":-0.9, "e":-0.5},
    {"id":4, "name":"EconCrisis",     "b":-0.25,"d":0.10,"m":-0.4, "e":-0.4},
    {"id":5, "name":"Climate",        "b":-0.10,"d":0.20,"m":-0.5, "e":-0.25},
    {"id":6, "name":"TechDisrupt",    "b":-0.12,"d":0.05,"m":-0.3, "e":-0.35},
    {"id":7, "name":"EnergyCrisis",   "b":-0.18,"d":0.30,"m":-0.7, "e":-0.45},
    {"id":8, "name":"SocialUpheaval", "b":-0.22,"d":0.12,"m":-0.6, "e":-0.28},
    {"id":9, "name":"InfraCollapse",  "b":-0.35,"d":0.50,"m":-0.8, "e":-0.60},
    {"id":10,"name":"FoodCrisis",     "b":-0.28,"d":0.35,"m":-0.75,"e":-0.40},
]


class RussiaRegionsEnv:
    """
    Среда MARL для всех субъектов РФ.
    - N агентов = N регионов (parameter sharing)
    - State: [14 числовых признаков + 9 ФО one-hot]
    - Action: 9 политик в [0,1], sum ≤ BUDGET_LIMIT
    - Переходная динамика: EnsembleWorldModel + граф миграции
    - Кризисы: стохастические шоки с региональной уязвимостью
    """
    def __init__(self, world_model, initial_states, adj_mat,
                 vuln_map, region_list, cfg, feat_cols):
        self.wm       = world_model
        self.adj      = torch.tensor(adj_mat,dtype=torch.float32,device=DEVICE)
        self.n        = len(region_list)
        self.s_dim    = cfg.STATE_DIM
        self.a_dim    = cfg.ACTION_DIM
        self.cfg      = cfg
        self.cols     = feat_cols
        self.regions  = region_list
        self.vuln     = np.array([vuln_map.get(r,0.5) for r in region_list],dtype=np.float32)
        self.ep_len   = cfg.MADDPG_EPISODE_LEN
        self.states   = initial_states.copy()
        self._step    = 0
        self._crisis  = None
        self._crisis_t= 0

        # Индексы ключевых признаков для reward
        fi = {f:i for i,f in enumerate(feat_cols)}
        self.i_pop  = fi.get("population",0)
        self.i_le   = fi.get("life_expectancy",1)
        self.i_tfr  = fi.get("tfr",2)
        self.i_mig  = fi.get("migration_balance_rate",7)
        self.i_pov  = fi.get("poverty_rate",6)
        self.i_un   = fi.get("unemployment",4)

    def reset(self, init=None):
        self.states  = (init if init is not None else self.states).copy()
        self._step   = 0
        self._crisis = None
        self._crisis_t = 0
        return self.states.copy()

    def _crisis_ctx(self):
        if self._crisis_t > 0:
            self._crisis_t -= 1
            c = self._crisis
            return np.array([c["b"],c["d"],c["m"],c["e"]],dtype=np.float32), c

        r = np.random.random()
        if   r < self.cfg.TAIL_RISK_PROB:
            c = CRISIS_SCENARIOS[np.random.randint(6,10)]
        elif r < self.cfg.CRISIS_INJECT_PROB:
            c = CRISIS_SCENARIOS[np.random.randint(0,6)]
        else:
            return np.zeros(4,dtype=np.float32), None

        self._crisis   = c
        self._crisis_t = np.random.randint(1,4)
        return np.array([c["b"],c["d"],c["m"],c["e"]],dtype=np.float32), c

    def step(self, actions):
        actions = np.clip(actions,0,1)
        # Бюджетное ограничение
        budget = actions.sum(axis=1,keepdims=True)
        scale  = np.where(budget>self.cfg.BUDGET_LIMIT,
                          self.cfg.BUDGET_LIMIT/(budget+1e-8), 1.0)
        actions = actions * scale

        ctx_np, c_obj = self._crisis_ctx()
        c_batch = (np.outer(self.vuln,ctx_np) if c_obj is not None
                   else np.zeros((self.n,4),dtype=np.float32))

        s_t  = torch.tensor(self.states, dtype=torch.float32, device=DEVICE)
        a_t  = torch.tensor(actions,     dtype=torch.float32, device=DEVICE)
        n_t  = (self.adj @ s_t)           # граф-агрегация
        c_t  = torch.tensor(c_batch,      dtype=torch.float32, device=DEVICE)

        self.wm.eval()
        with torch.no_grad():
            delta_mean, uncert = self.wm(s_t,a_t,n_t,c_t)
            noise = torch.randn_like(delta_mean)*uncert*0.05
            next_s = (s_t + delta_mean + noise).cpu().numpy()

        rewards = self._reward(self.states, next_s, actions, c_obj)
        self.states = next_s
        self._step += 1
        done = self._step >= self.ep_len
        info = {"crisis": c_obj["name"] if c_obj else None,
                "unc":    uncert.mean().item()}
        return next_s, rewards, done, info

    def _reward(self, sp, sn, a, c_obj):
        W  = self.cfg.REWARD_WEIGHTS
        def g(x,i): return x[:,i] if i<x.shape[1] else np.zeros(x.shape[0])
        t = np.tanh
        r = (W["population"]    * t((g(sn,self.i_pop) -g(sp,self.i_pop)) *3) +
             W["life_expectancy"]* t((g(sn,self.i_le)  -g(sp,self.i_le))  *3) +
             W["fertility"]      * t((g(sn,self.i_tfr) -g(sp,self.i_tfr)) *3) +
             W["migration"]      * t((g(sn,self.i_mig) -g(sp,self.i_mig)) *3) +
             W["poverty"]        * t(-(g(sn,self.i_pov)-g(sp,self.i_pov)) *3) +
             W["unemployment"]   * t(-(g(sn,self.i_un) -g(sp,self.i_un))  *3) +
             W["budget"]         * (-np.maximum(0, a.sum(1)-self.cfg.BUDGET_LIMIT)))
        return r.astype(np.float32)


def build_init_states(df_split, regions, state_cols, year=None):
    y = year if year else df_split["year"].max()
    dfy = df_split[df_split["year"]==y].set_index("region")
    st  = np.zeros((len(regions),len(state_cols)),dtype=np.float32)
    for i,r in enumerate(regions):
        if r in dfy.index:
            st[i] = np.nan_to_num(dfy.loc[r,state_cols].values.astype(float))
    return st


init_tr = build_init_states(df_train, regions, ALL_STATE_COLS)
init_te = build_init_states(df_test if len(df_test)>0 else df_norm,
                             regions, ALL_STATE_COLS)

env_train = RussiaRegionsEnv(world_model,init_tr,adj_norm_mat,
                              VULN_MAP,regions,CFG,ALL_STATE_COLS)
env_test  = RussiaRegionsEnv(world_model,init_te,adj_norm_mat,
                              VULN_MAP,regions,CFG,ALL_STATE_COLS)

print(f"Среда RussiaRegionsEnv:")
print(f"  Агентов: {env_train.n}")
print(f"  State dim: {env_train.s_dim}  Action dim: {env_train.a_dim}")
print(f"  Episode length: {env_train.ep_len} шагов")
print(f"  Budget limit: {CFG.BUDGET_LIMIT}")
print("✅ MARL Среда готова")


Среда RussiaRegionsEnv:
  Агентов: 82
  State dim: 10  Action dim: 9
  Episode length: 15 шагов
  Budget limit: 3.5
✅ MARL Среда готова


In [13]:
# ── [13] MADDPG: SharedActor / SharedCritic / ReplayBuffer / Обучение ─────────

class SharedActor(nn.Module):
    """Детерминированный актор с parameter sharing (одна сеть на всех агентов)."""
    def __init__(self, s_dim, a_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim,hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden,hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden,a_dim), nn.Sigmoid()          # ∈ [0,1]
        )
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.orthogonal_(m.weight,gain=0.01)
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self,s): return self.net(s)


class SharedCritic(nn.Module):
    """Централизованный критик: state_i || action_i || neighbor_agg."""
    def __init__(self, s_dim, a_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim*2+a_dim,hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden,hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden,1)
        )
    def forward(self,s,a,ns): return self.net(torch.cat([s,a,ns],dim=-1))


class ReplayBuffer:
    """Кольцевой буфер для off-policy обучения."""
    def __init__(self, cap, n, s_dim, a_dim):
        self.cap=cap; self.n=n; self.ptr=0; self.size=0
        self.S =np.zeros((cap,n,s_dim),dtype=np.float32)
        self.A =np.zeros((cap,n,a_dim),dtype=np.float32)
        self.R =np.zeros((cap,n),      dtype=np.float32)
        self.S1=np.zeros((cap,n,s_dim),dtype=np.float32)
        self.D =np.zeros((cap,),       dtype=np.float32)

    def push(self,s,a,r,s1,done):
        self.S[self.ptr]=s; self.A[self.ptr]=a
        self.R[self.ptr]=r; self.S1[self.ptr]=s1
        self.D[self.ptr]=float(done)
        self.ptr=(self.ptr+1)%self.cap
        self.size=min(self.size+1,self.cap)

    def sample(self,B):
        idx=np.random.randint(0,self.size,B)
        to = lambda x: torch.tensor(x[idx],device=DEVICE)
        return to(self.S),to(self.A),to(self.R),to(self.S1),to(self.D)
    def __len__(self): return self.size


def ou_noise(a,sigma=0.15): return (a+sigma*torch.randn_like(a)).clamp(0,1)

def neigh_agg(states,adj):
    """Batch-wise matrix multiply: [B,N,S] x adj[N,N] → [B,N,S]."""
    return torch.bmm(adj.unsqueeze(0).expand(states.shape[0],-1,-1),states)


def train_maddpg(env, cfg, logger):
    """Полный цикл MADDPG с ранней остановкой."""
    N,S,A = env.n, env.s_dim, env.a_dim
    actor  = SharedActor(S,A,cfg.MADDPG_HIDDEN).to(DEVICE)
    critic = SharedCritic(S,A,cfg.MADDPG_HIDDEN).to(DEVICE)
    t_a    = copy.deepcopy(actor).to(DEVICE)
    t_c    = copy.deepcopy(critic).to(DEVICE)
    for p in list(t_a.parameters())+list(t_c.parameters()):
        p.requires_grad=False

    opt_a = Adam(actor.parameters(), lr=cfg.MADDPG_LR_ACTOR,  weight_decay=1e-5)
    opt_c = Adam(critic.parameters(),lr=cfg.MADDPG_LR_CRITIC, weight_decay=1e-5)
    buf   = ReplayBuffer(cfg.MADDPG_BUFFER_SIZE,N,S,A)
    es    = EarlyStopping(patience=cfg.RL_PATIENCE,mode="max",restore_best=True)
    adj_t = ADJ_TENSOR
    sigma = 0.20; total_steps=0; best_rew=-np.inf
    loss_c_v=0.0

    for ep in trange(cfg.MADDPG_EPISODES,desc="MADDPG",leave=True):
        states=env.reset(init_tr.copy()); ep_rew=0.0
        for _ in range(cfg.MADDPG_EPISODE_LEN):
            s_t=torch.tensor(states,dtype=torch.float32,device=DEVICE)
            actor.eval()
            with torch.no_grad():
                acts=ou_noise(actor(s_t),sigma) if total_steps>=cfg.MADDPG_WARMUP                      else torch.rand(N,A,device=DEVICE)
            nxt,rws,done,_ = env.step(acts.cpu().numpy())
            buf.push(states,acts.cpu().numpy(),rws,nxt,done)
            ep_rew+=rws.mean(); states=nxt; total_steps+=1

            if len(buf)<cfg.MADDPG_BATCH_SIZE or total_steps<cfg.MADDPG_WARMUP:
                if done: break; continue

            Sb,Ab,Rb,S1b,Db = buf.sample(cfg.MADDPG_BATCH_SIZE)
            ns=neigh_agg(Sb,adj_t); ns1=neigh_agg(S1b,adj_t)
            B2=Sb.shape[0]

            with torch.no_grad():
                A1=t_a(S1b.view(-1,S))
                qt=t_c(S1b.view(-1,S),A1,ns1.view(-1,S)).view(B2,N)
                td=Rb+cfg.MADDPG_GAMMA*qt*(1-Db.unsqueeze(1))

            qp=critic(Sb.view(-1,S),Ab.view(-1,A),ns.view(-1,S)).view(B2,N)
            lc=F.mse_loss(qp,td.detach())
            opt_c.zero_grad(); lc.backward()
            nn.utils.clip_grad_norm_(critic.parameters(),1.0)
            opt_c.step(); loss_c_v=lc.item()

            actor.train()
            ac=actor(Sb.view(-1,S))
            la=-critic(Sb.view(-1,S),ac,ns.view(-1,S).detach()).mean()
            opt_a.zero_grad(); la.backward()
            nn.utils.clip_grad_norm_(actor.parameters(),1.0)
            opt_a.step()

            tau=cfg.MADDPG_TAU
            for p,tp in zip(actor.parameters(),t_a.parameters()):
                tp.data.copy_(tau*p.data+(1-tau)*tp.data)
            for p,tp in zip(critic.parameters(),t_c.parameters()):
                tp.data.copy_(tau*p.data+(1-tau)*tp.data)
            if done: break

        sigma=max(0.02,sigma*0.995)
        logger.log(ep,{"reward":ep_rew,"sigma":sigma,"loss_c":loss_c_v})
        if ep_rew>best_rew:
            best_rew=ep_rew
            torch.save(actor.state_dict(),"output/checkpoints/maddpg_actor_best.pt")
        if ep%cfg.MADDPG_LOG_INTERVAL==0:
            tqdm.write(f"  MADDPG ep={ep:4d}  rew={ep_rew:.4f}  σ={sigma:.3f}")
        if es.step(ep_rew) and ep>150:
            tqdm.write(f"  MADDPG early stop ep={ep}"); break

    logger.save()
    actor.load_state_dict(torch.load("output/checkpoints/maddpg_actor_best.pt",
                                      map_location=DEVICE))
    return actor, critic


with Timer("Обучение MADDPG"):
    maddpg_log = ExperimentLogger("MADDPG")
    maddpg_actor, maddpg_critic = train_maddpg(env_train,CFG,maddpg_log)

print("✅ MADDPG обучен")


MADDPG:   0%|          | 1/1000 [00:00<11:13,  1.48it/s]

  MADDPG ep=   0  rew=0.8947  σ=0.199


MADDPG:   5%|▌         | 51/1000 [00:18<06:17,  2.52it/s]

  MADDPG ep=  50  rew=1.8904  σ=0.155


MADDPG:  10%|█         | 101/1000 [00:42<05:52,  2.55it/s]

...

  MADDPG ep= 250  rew=2.6430  σ=0.057


MADDPG:  30%|███       | 301/1000 [02:05<04:40,  2.49it/s]

  MADDPG ep= 300  rew=2.5937  σ=0.044


MADDPG:  30%|███       | 302/1000 [02:06<04:51,  2.40it/s]

  MADDPG early stop ep=302
  ⏱  Обучение MADDPG: 126.1s
✅ MADDPG обучен


In [14]:
# ── [14] MATD3: Twin Critic + Policy Delay + Target Policy Noise ───────────────

class TwinCritic(nn.Module):
    """Два независимых Q-критика для снижения overestimation bias."""
    def __init__(self, s_dim, a_dim, hidden=256):
        super().__init__()
        in_d = s_dim*2+a_dim
        def _mlp():
            return nn.Sequential(
                nn.Linear(in_d,hidden), nn.LayerNorm(hidden), nn.GELU(),
                nn.Linear(hidden,hidden), nn.LayerNorm(hidden), nn.GELU(),
                nn.Linear(hidden,1)
            )
        self.q1=_mlp(); self.q2=_mlp()

    def forward(self,s,a,ns):
        x=torch.cat([s,a,ns],dim=-1)
        return self.q1(x), self.q2(x)

    def q1_only(self,s,a,ns):
        return self.q1(torch.cat([s,a,ns],dim=-1))


def train_matd3(env, cfg, logger):
    """MATD3 с double critic, policy delay и target policy smoothing."""
    N,S,A = env.n, env.s_dim, env.a_dim
    actor  = SharedActor(S,A,cfg.MATD3_HIDDEN).to(DEVICE)
    critic = TwinCritic(S,A,cfg.MATD3_HIDDEN).to(DEVICE)
    t_a    = copy.deepcopy(actor).to(DEVICE)
    t_c    = copy.deepcopy(critic).to(DEVICE)
    for p in list(t_a.parameters())+list(t_c.parameters()):
        p.requires_grad=False

    opt_a = Adam(actor.parameters(), lr=cfg.MATD3_LR_ACTOR,  weight_decay=1e-5)
    opt_c = Adam(critic.parameters(),lr=cfg.MATD3_LR_CRITIC, weight_decay=1e-5)
    buf   = ReplayBuffer(cfg.MADDPG_BUFFER_SIZE,N,S,A)
    es    = EarlyStopping(patience=cfg.RL_PATIENCE,mode="max",restore_best=True)
    adj_t = ADJ_TENSOR
    sigma=0.20; total_steps=0; a_step=0; best_rew=-np.inf
    lc_v=0.0; la_v=0.0

    for ep in trange(cfg.MATD3_EPISODES,desc="MATD3",leave=True):
        states=env.reset(init_tr.copy()); ep_rew=0.0
        for _ in range(cfg.MADDPG_EPISODE_LEN):
            s_t=torch.tensor(states,dtype=torch.float32,device=DEVICE)
            actor.eval()
            with torch.no_grad():
                acts=ou_noise(actor(s_t),sigma) if total_steps>=cfg.MADDPG_WARMUP                      else torch.rand(N,A,device=DEVICE)
            nxt,rws,done,_=env.step(acts.cpu().numpy())
            buf.push(states,acts.cpu().numpy(),rws,nxt,done)
            ep_rew+=rws.mean(); states=nxt; total_steps+=1

            if len(buf)<cfg.MADDPG_BATCH_SIZE or total_steps<cfg.MADDPG_WARMUP:
                if done: break; continue

            Sb,Ab,Rb,S1b,Db=buf.sample(cfg.MADDPG_BATCH_SIZE)
            ns=neigh_agg(Sb,adj_t); ns1=neigh_agg(S1b,adj_t); B2=Sb.shape[0]

            with torch.no_grad():
                A1=t_a(S1b.view(-1,S))
                noise=(torch.randn_like(A1)*cfg.MATD3_TARGET_NOISE
                       ).clamp(-cfg.MATD3_NOISE_CLIP,cfg.MATD3_NOISE_CLIP)
                A1=(A1+noise).clamp(0,1)
                q1t,q2t=t_c(S1b.view(-1,S),A1,ns1.view(-1,S))
                td=Rb+cfg.MATD3_GAMMA*torch.min(q1t,q2t).view(B2,N)*(1-Db.unsqueeze(1))

            q1p,q2p=critic(Sb.view(-1,S),Ab.view(-1,A),ns.view(-1,S))
            lc=(F.mse_loss(q1p.view(B2,N),td.detach())+
                F.mse_loss(q2p.view(B2,N),td.detach()))
            opt_c.zero_grad(); lc.backward()
            nn.utils.clip_grad_norm_(critic.parameters(),1.0)
            opt_c.step(); lc_v=lc.item(); a_step+=1

            # Delayed actor update
            if a_step%cfg.MATD3_POLICY_DELAY==0:
                actor.train()
                ac=actor(Sb.view(-1,S))
                la=-critic.q1_only(Sb.view(-1,S),ac,ns.view(-1,S).detach()).mean()
                opt_a.zero_grad(); la.backward()
                nn.utils.clip_grad_norm_(actor.parameters(),1.0)
                opt_a.step(); la_v=la.item()
                tau=cfg.MATD3_TAU
                for p,tp in zip(actor.parameters(),t_a.parameters()):
                    tp.data.copy_(tau*p.data+(1-tau)*tp.data)
                for p,tp in zip(critic.parameters(),t_c.parameters()):
                    tp.data.copy_(tau*p.data+(1-tau)*tp.data)
            if done: break

        sigma=max(0.02,sigma*0.995)
        logger.log(ep,{"reward":ep_rew,"loss_c":lc_v,"loss_a":la_v})
        if ep_rew>best_rew:
            best_rew=ep_rew
            torch.save(actor.state_dict(),"output/checkpoints/matd3_actor_best.pt")
        if ep%cfg.MATD3_LOG_INTERVAL==0:
            tqdm.write(f"  MATD3  ep={ep:4d}  rew={ep_rew:.4f}")
        if es.step(ep_rew) and ep>150:
            tqdm.write(f"  MATD3  early stop ep={ep}"); break

    logger.save()
    actor.load_state_dict(torch.load("output/checkpoints/matd3_actor_best.pt",
                                      map_location=DEVICE))
    return actor, critic


with Timer("Обучение MATD3"):
    matd3_log = ExperimentLogger("MATD3")
    matd3_actor, matd3_critic = train_matd3(env_train,CFG,matd3_log)

print("✅ MATD3 обучен")


MATD3:   0%|          | 1/1000 [00:00<07:45,  2.14it/s]

  MATD3  ep=   0  rew=0.9005


MATD3:   5%|▌         | 51/1000 [00:22<07:14,  2.18it/s]

  MATD3  ep=  50  rew=2.1894


MATD3:  10%|█         | 101/1000 [00:45<06:51,  2.18it/s]

...

  MATD3  ep= 400  rew=2.7465


MATD3:  45%|████▌     | 451/1000 [03:25<04:12,  2.17it/s]

  MATD3  ep= 450  rew=2.7287


MATD3:  47%|████▋     | 466/1000 [03:32<04:03,  2.19it/s]

  MATD3  early stop ep=466
  ⏱  Обучение MATD3: 212.9s
✅ MATD3 обучен


In [15]:
# ── [15] Evo-MATD3-SR (action-bias offline fine-tuning) ──────────────────────

class BiasWrapper(nn.Module):
    """
    Обёртка над уже обученным MATD3-actor:
    a(s) = clip( base_actor(s) + b, 0, 1 ),
    где b — маленький вектор смещений по действиям (эволюционируемый).
    Архитектура base_actor и reward-функция среды НЕ меняются.
    """
    def __init__(self, base_actor: nn.Module, action_dim: int, init_bias: float = 0.0):
        super().__init__()
        self.base_actor = base_actor
        self.bias = nn.Parameter(torch.ones(action_dim) * init_bias, requires_grad=False)

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        # Гарантируем единый device
        dev = s.device
        self.base_actor.to(dev)
        if self.bias.device != dev:
            self.bias.data = self.bias.data.to(dev)

        with torch.no_grad():
            a = self.base_actor(s)  # (B, A)
            b = self.bias.view(1, -1).expand_as(a)
            a_biased = torch.clamp(a + b, 0.0, 1.0)
        return a_biased


def evaluate_bias_candidate(
    bias_vec: np.ndarray,
    base_actor: nn.Module,
    env: RussiaRegionsEnv,
    init_states: np.ndarray,
    n_episodes: int = 5,
    max_len: int = None,
) -> float:
    """
    Fitness = средний эпизодический reward на train-env
    для политики base_actor + bias_vec.
    Reward, env, архитектура — те же, что у MATD3.
    """
    actor = BiasWrapper(base_actor, action_dim=env.a_dim)
    with torch.no_grad():
        actor.bias.data.copy_(torch.tensor(bias_vec, dtype=torch.float32, device=DEVICE))
    actor.to(DEVICE)

    if max_len is None:
        max_len = getattr(env, "ep_len", None)
        if max_len is None:
            max_len = getattr(env, "eplen", None)
        if max_len is None:
            max_len = getattr(env, "episode_len", None)
        if max_len is None:
            max_len = CFG.MADDPG_EPISODE_LEN

    rewards = []
    for _ in range(n_episodes):
        states = env.reset(init_states.copy())
        ep_rew = 0.0
        for _ in range(max_len):
            s_t = torch.tensor(states, dtype=torch.float32, device=DEVICE)
            a = actor(s_t).cpu().numpy()
            states, rws, done, _ = env.step(a)
            ep_rew += float(rws.mean())
            if done:
                break
        rewards.append(ep_rew)

    return float(np.mean(rewards))


def evo_tune_action_bias(
    base_actor: nn.Module,
    env: RussiaRegionsEnv,
    init_states: np.ndarray,
    cfg: Config,
    pop_size: int = 16,
    elite_frac: float = 0.3,
    n_iters: int = 60,
    bias_std_init: float = 0.04,
    bias_std_min: float = 0.005,
    bias_std_decay: float = 0.97,
    eval_episodes_fast: int = 3,
    eval_episodes_refine: int = 6,
):
    """
    Offline-эволюция маленького bias-вектора b (R^{A}):
    - популяция size=pop_size;
    - мутация ~ N(0, sigma) с затухающим sigma;
    - fitness = средний эпизодический reward env (тот же, что у MATD3);
    - никаких изменений архитектуры и reward-функции.
    """
    A = env.a_dim
    mu = np.zeros(A, dtype=np.float32)
    sigma = np.ones(A, dtype=np.float32) * bias_std_init
    elite_n = max(1, int(pop_size * elite_frac))

    records = []

    # глобально лучший bias по refined-score
    global_best_bias = mu.copy()
    global_best_score = -np.inf

    for it in range(n_iters):
        cand = []
        scores = []

        # fast-оценка популяции
        for k in range(pop_size):
            eps = np.random.randn(A).astype(np.float32) * sigma
            bias_k = mu + eps
            score_k = evaluate_bias_candidate(
                bias_k, base_actor, env, init_states, n_episodes=eval_episodes_fast
            )
            cand.append(bias_k)
            scores.append(score_k)

        scores_np = np.array(scores, dtype=np.float32)
        elite_idx = np.argsort(scores_np)[-elite_n:][::-1]

        # refine top-k кандидатов на большем числе эпизодов
        for i in elite_idx:
            score_ref = evaluate_bias_candidate(
                cand[i], base_actor, env, init_states, n_episodes=eval_episodes_refine
            )
            scores_np[i] = (scores_np[i] + score_ref) / 2.0

        # переоценка элит после refine
        elite_idx = np.argsort(scores_np)[-elite_n:][::-1]
        elites = np.stack([cand[i] for i in elite_idx], axis=0)

        # обновляем глобально лучший bias по refined-score
        it_best_idx = int(np.argmax(scores_np))
        it_best_score = float(scores_np[it_best_idx])
        if it_best_score > global_best_score:
            global_best_score = it_best_score
            global_best_bias = cand[it_best_idx].copy()

        # обновление распределения (mu, sigma) как в ES/CEM
        mu = elites.mean(axis=0)
        elite_std = elites.std(axis=0)
        sigma = (sigma * bias_std_decay + elite_std * (1.0 - bias_std_decay)).clip(
            bias_std_min, None
        )

        records.append({
            "iter": it,
            "score_mean": float(scores_np.mean()),
            "score_std": float(scores_np.std()),
            "score_best": float(scores_np.max()),
            "sigma_mean": float(sigma.mean()),
        })

        if (it + 1) % 5 == 0:
            print(
                f"[Evo-Bias] it={it+1:02d} "
                f"reward_mean={scores_np.mean():.4f} "
                f"reward_best={scores_np.max():.4f} "
                f"sigma={sigma.mean():.4f}"
            )

    # финальный bias: глобально лучший по refined-score
    best_bias = global_best_bias.astype(np.float32)

    evo_actor = BiasWrapper(base_actor, action_dim=env.a_dim)
    evo_actor.to(DEVICE)
    with torch.no_grad():
        evo_actor.bias.data.copy_(torch.tensor(best_bias, dtype=torch.float32, device=DEVICE))

    evo_log = pd.DataFrame(records)
    return evo_actor, evo_log


with Timer("Evo-MATD3-SR (action-bias offline fine-tuning)"):
    # Загружаем уже обученный MATD3-actor (baseline)
    matd3_actor = SharedActor(env_train.s_dim, env_train.a_dim, CFG.MATD3_HIDDEN).to(DEVICE)
    matd3_actor.load_state_dict(
        torch.load("output/checkpoints/matd3_actor_best.pt", map_location=DEVICE)
    )

    evo_actor, evo_bias_log = evo_tune_action_bias(
        base_actor=matd3_actor,
        env=env_train,
        init_states=init_tr,
        cfg=CFG,
        pop_size=16,
        elite_frac=0.3,
        n_iters=60,
        bias_std_init=0.04,
        bias_std_min=0.005,
        bias_std_decay=0.97,
        eval_episodes_fast=3,
        eval_episodes_refine=6,
    )

    AM.save_table(evo_bias_log, "15_evo_bias_log")
    torch.save(evo_actor.state_dict(), "output/checkpoints/evomatd3sr_bias_actor.pt")

print("✅ Evo-MATD3-SR (bias) обучен")

[Evo-Bias] it=05 reward_mean=2.8230 reward_best=2.8608 sigma=0.0386
[Evo-Bias] it=10 reward_mean=2.8556 reward_best=2.8701 sigma=0.0367
[Evo-Bias] it=15 reward_mean=2.8575 reward_best=2.8749 sigma=0.0355
[Evo-Bias] it=20 reward_mean=2.8650 reward_best=2.8766 sigma=0.0344
[Evo-Bias] it=25 reward_mean=2.8607 reward_best=2.8858 sigma=0.0329
[Evo-Bias] it=30 reward_mean=2.8569 reward_best=2.8808 sigma=0.0313
[Evo-Bias] it=35 reward_mean=2.8645 reward_best=2.8771 sigma=0.0305
[Evo-Bias] it=40 reward_mean=2.8631 reward_best=2.8877 sigma=0.0294
[Evo-Bias] it=45 reward_mean=2.8637 reward_best=2.8825 sigma=0.0283
[Evo-Bias] it=50 reward_mean=2.8667 reward_best=2.8734 sigma=0.0272
[Evo-Bias] it=55 reward_mean=2.8629 reward_best=2.8778 sigma=0.0264
[Evo-Bias] it=60 reward_mean=2.8677 reward_best=2.8845 sigma=0.0257
  ⏱  Evo-MATD3-SR (action-bias offline fine-tuning): 187.2s
✅ Evo-MATD3-SR (bias) обучен


In [16]:
# ── [16] Сравнительная оценка алгоритмов на тестовой среде ────────────────────

def evaluate_agent(actor, env, init_states, n_episodes=5, label="Agent"):
    """Прогоняет агента на test-среде и собирает метрики."""
    all_rewards = []
    all_actions = []

    ep_len = getattr(env, "ep_len", None)
    if ep_len is None:
        ep_len = getattr(env, "eplen", None)
    if ep_len is None:
        ep_len = getattr(env, "episode_len", None)
    if ep_len is None:
        ep_len = CFG.MADDPG_EPISODE_LEN

    for ep in range(n_episodes):
        states = env.reset(init_states.copy())
        ep_rew = 0.0
        ep_acts = []

        for _ in range(ep_len):
            s_t = torch.tensor(states, dtype=torch.float32, device=DEVICE)
            actor.eval()
            with torch.no_grad():
                acts = actor(s_t).cpu().numpy()

            states, rws, done, _ = env.step(acts)
            ep_rew += rws.mean()
            ep_acts.append(acts.mean(axis=0))

            if done:
                break

        all_rewards.append(ep_rew)
        all_actions.extend(ep_acts)

    return {
        "label": label,
        "mean_rew": np.mean(all_rewards),
        "std_rew": np.std(all_rewards),
        "mean_act": np.mean(all_actions, axis=0) if all_actions else np.zeros(CFG.ACTION_DIM)
    }


# Инерционный baseline (нулевые действия)
class ZeroActor(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.a = a_dim

    def forward(self, s):
        return torch.zeros(s.shape[0], self.a, device=s.device)


zero_actor = ZeroActor(CFG.STATE_DIM, CFG.ACTION_DIM).to(DEVICE)

# минимальное изменение: грузим сохранённые MATD3 и offline Evo actor
matd3_actor = SharedActor(env_test.s_dim, env_test.a_dim, CFG.MATD3_HIDDEN).to(DEVICE)
matd3_actor.load_state_dict(
    torch.load("output/checkpoints/matd3_actor_best.pt", map_location=DEVICE)
)

evo_actor = BiasWrapper(matd3_actor, action_dim=env_test.a_dim)
evo_actor.load_state_dict(
    torch.load("output/checkpoints/evomatd3sr_bias_actor.pt", map_location=DEVICE)
)
evo_actor.to(DEVICE)
print("Оценка на test-среде (5 эпизодов каждый)...")
results = []
for actor_obj, label in [
    (zero_actor,   "Baseline (no action)"),
    (maddpg_actor, "MADDPG"),
    (matd3_actor,  "MATD3"),
    (evo_actor,    "Evo-MATD3-SR"),
]:
    r = evaluate_agent(actor_obj, env_test, init_te, n_episodes=5, label=label)
    results.append(r)
    print(f"  {label:25s}  reward={r['mean_rew']:.4f} ± {r['std_rew']:.4f}")


# Сводная таблица
res_df = pd.DataFrame([{
    "Алгоритм": r["label"],
    "Mean Reward": round(r["mean_rew"], 4),
    "Std Reward": round(r["std_rew"], 4),
} for r in results])
AM.save_table(res_df, "06_algorithm_comparison")
print("\nСводная таблица:")
print(res_df.to_string(index=False))


# ── Визуализация сравнения ────────────────────────────────────────────────────
labels = [r["label"] for r in results]
means  = [r["mean_rew"] for r in results]
stds   = [r["std_rew"] for r in results]
colors = ["#aaaaaa", "#4c72b0", "#dd8452", "#55a868"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=5, color=colors, alpha=0.85, edgecolor="black")
ax.set_ylabel("Mean Episodic Reward")
ax.set_title("Сравнение алгоритмов MARL (test set, 5 эпизодов)")
ax.tick_params(axis="x", rotation=15)

for bar, m in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{m:.4f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.grid(True, alpha=0.4, axis="y")
plt.tight_layout()
AM.save_figure(fig, "05_algorithm_comparison")
print("  → output/figures/05_algorithm_comparison.png")
print("✅ Сравнительная оценка завершена")

Оценка на test-среде (5 эпизодов каждый)...
  Baseline (no action)       reward=-0.3044 ± 0.0407
  MADDPG                     reward=2.0044 ± 0.0311
  MATD3                      reward=2.2022 ± 0.0189
  Evo-MATD3-SR               reward=2.2331 ± 0.0063

Сводная таблица:
            Алгоритм  Mean Reward  Std Reward
Baseline (no action)      -0.3044      0.0407
              MADDPG       2.0044      0.0311
               MATD3       2.2022      0.0189
        Evo-MATD3-SR       2.2331      0.0063
  → output/figures/05_algorithm_comparison.png
✅ Сравнительная оценка завершена


In [17]:
# ── [17] Кривые обучения всех алгоритмов ──────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Кривые обучения RL-агентов", fontsize=12, fontweight="bold")

# Обновлённые ссылки на логи:
# maddpg_log  — как раньше (ячейка 13)
# matd3_log   — как раньше (ячейка 14)
# evo_bias_log — лог из новой 15-й (Evo-MATD3-SR bias)

algo_logs = [
    ("MADDPG",          maddpg_log,   "steelblue"),
    ("MATD3",           matd3_log,    "darkorange"),
    ("Evo-MATD3-SR\n(bias)", evo_bias_log, "green"),
]

for i, (name, lg, col) in enumerate(algo_logs):
    # если лог не определён или пустой — пропускаем
    if lg is None:
        continue
    df_lg = lg if isinstance(lg, pd.DataFrame) else lg.to_df()
    if df_lg is None or df_lg.empty or "reward" not in df_lg.columns:
        continue

    ax = axes[i]

    raw = df_lg["reward"].values
    # Скользящее среднее (окно 20 или меньше, если эпизодов мало)
    window = min(20, max(1, len(raw) // 4 + 1))
    smooth = pd.Series(raw).rolling(window, min_periods=1).mean().values

    ax.plot(df_lg["episode"], raw, alpha=0.3, color=col, lw=0.8, label="raw")
    ax.plot(df_lg["episode"], smooth, color=col, lw=2.0, label=f"{name} (MA)")
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Эпизод")
    ax.set_ylabel("Reward")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
AM.save_figure(fig, "06_training_curves")
print("  → output/figures/06_training_curves.png")
print("✅ Кривые обучения сохранены")

  → output/figures/06_training_curves.png
✅ Кривые обучения сохранены


In [18]:
# ── [18] Стресс-тест: реакция агентов на различные кризисы ────────────────────

def crisis_stress_test(actor, env, init_states, crisis_scenarios, n_reps=3):
    """Фиксирует reward при принудительном включении каждого кризиса."""
    records = []

    # определяем длину эпизода безопасно
    ep_len = getattr(env, "ep_len", None)
    if ep_len is None:
        ep_len = getattr(env, "eplen", None)
    if ep_len is None:
        ep_len = getattr(env, "episode_len", None)
    if ep_len is None:
        ep_len = CFG.MADDPG_EPISODE_LEN

    original_p = env.cfg.CRISIS_INJECT_PROB
    env.cfg.CRISIS_INJECT_PROB = 1.0  # всегда кризис

    for c in crisis_scenarios:
        rew_list = []
        for _ in range(n_reps):
            # принудительно задаём сценарий кризиса
            env._crisis = c
            env._crisis_t = ep_len
            states = env.reset(init_states.copy())
            ep_rew = 0.0

            for _ in range(ep_len):
                s_t = torch.tensor(states, dtype=torch.float32, device=DEVICE)
                actor.eval()
                with torch.no_grad():
                    acts = actor(s_t).cpu().numpy()
                states, rws, done, _ = env.step(acts)
                ep_rew += rws.mean()
                if done:
                    break

            rew_list.append(ep_rew)

        records.append({
            "crisis":   c["name"],
            "mean_rew": np.mean(rew_list),
            "std_rew":  np.std(rew_list),
        })

    env.cfg.CRISIS_INJECT_PROB = original_p
    return pd.DataFrame(records)


print("Стресс-тест по кризисным сценариям...")

# здесь используем те же акторы, что и в ячейке 16:
# maddpg_actor  — из ячейки 13
# matd3_actor   — уже загружен в 16-й из чекпойнта
# evo_actor     — BiasWrapper(matd3_actor, ...) с bias-чекпойнтом

stress_results = {}
for actor_obj, label in [
    (maddpg_actor, "MADDPG"),
    (matd3_actor,  "MATD3"),
    (evo_actor,    "Evo-MATD3-SR"),
]:
    df_s = crisis_stress_test(actor_obj, env_test, init_te, CRISIS_SCENARIOS[:6], n_reps=3)
    df_s["algorithm"] = label
    stress_results[label] = df_s
    print(f"  {label}")

df_stress_all = pd.concat(stress_results.values(), ignore_index=True)
AM.save_table(df_stress_all, "07_crisis_stress_test")

# Тепловая карта
stress_pivot = df_stress_all.pivot(index="crisis", columns="algorithm", values="mean_rew")
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    stress_pivot,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Mean Reward"},
)
ax.set_title("Стресс-тест: mean reward при различных кризисах", fontsize=12, fontweight="bold")
ax.set_xlabel("Алгоритм")
ax.set_ylabel("Тип кризиса")
plt.tight_layout()
AM.save_figure(fig, "07_crisis_stress_heatmap")
print("  → output/figures/07_crisis_stress_heatmap.png")
print("✅ Стресс-тест завершён")

Стресс-тест по кризисным сценариям...
  MADDPG
  MATD3
  Evo-MATD3-SR
  → output/figures/07_crisis_stress_heatmap.png
✅ Стресс-тест завершён


In [19]:
# ── [19] Анализ политик: что выбирает лучший агент ────────────────────────────

def extract_policy_profiles(actor, env, init_states, n_steps=20):
    """Прогоняет агента и возвращает средний профиль действий по регионам."""
    # определяем длину эпизода безопасно
    ep_len = getattr(env, "ep_len", None)
    if ep_len is None:
        ep_len = getattr(env, "eplen", None)
    if ep_len is None:
        ep_len = getattr(env, "episode_len", None)
    if ep_len is None:
        ep_len = CFG.MADDPG_EPISODE_LEN

    states = env.reset(init_states.copy())
    all_acts = []
    for _ in range(min(n_steps, ep_len)):
        s_t = torch.tensor(states, dtype=torch.float32, device=DEVICE)
        actor.eval()
        with torch.no_grad():
            acts = actor(s_t).cpu().numpy()
        all_acts.append(acts.copy())
        states, _, done, _ = env.step(acts)
        if done:
            break

    mean_acts = np.mean(all_acts, axis=0)  # [N_regions, A_dim]
    return pd.DataFrame(mean_acts, index=env.regions, columns=CFG.ACTION_NAMES)


print("Извлечение профилей политик...")

# здесь evo_actor уже должен быть bias-версией из 15-й ячейки,
# как ты использовал его в 16-й (test-сравнение) и 18-й (стресс-тест)
policy_df = extract_policy_profiles(evo_actor, env_test, init_te, n_steps=20)
policy_df["cluster"]       = [CLUSTER_MAP.get(r, -1) for r in policy_df.index]
policy_df["vulnerability"] = [VULN_MAP.get(r, 0)    for r in policy_df.index]
AM.save_table(policy_df, "08_policy_profiles_evo")


# ── Радарные диаграммы по кластерам ──────────────────────────────────────────
cluster_mean = policy_df.groupby("cluster")[CFG.ACTION_NAMES].mean()
angles = np.linspace(0, 2 * np.pi, len(CFG.ACTION_NAMES), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(2, 3, figsize=(14, 9), subplot_kw={"projection": "polar"})
fig.suptitle(
    "Evo-MATD3-SR (bias): профили политик по кластерам регионов",
    fontsize=12,
    fontweight="bold",
)

pal_c = sns.color_palette("tab10", N_CLUSTERS)
for ci, ax in enumerate(axes.flat):
    if ci >= N_CLUSTERS:
        ax.set_visible(False)
        continue
    if ci not in cluster_mean.index:
        ax.set_visible(False)
        continue

    vals = cluster_mean.loc[ci].values.tolist()
    vals += vals[:1]

    ax.plot(angles, vals, color=pal_c[ci], lw=2)
    ax.fill(angles, vals, alpha=0.25, color=pal_c[ci])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([n.replace("_", "\n") for n in CFG.ACTION_NAMES], fontsize=6)
    ax.set_ylim(0, 1)
    n_regs = sum(1 for v in CLUSTER_MAP.values() if v == ci)
    ax.set_title(f"Кластер {ci+1} ({n_regs} регионов)", fontsize=9, fontweight="bold", pad=15)

plt.tight_layout()
AM.save_figure(fig, "08_policy_radar_clusters")
print("  → output/figures/08_policy_radar_clusters.png")


# ── Сравнение политик MADDPG vs MATD3 vs Evo ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Средние веса политик по 9 инструментам управления", fontsize=12, fontweight="bold")

for ax, actor_obj, label, col in zip(
    axes,
    [maddpg_actor, matd3_actor, evo_actor],
    ["MADDPG", "MATD3", "Evo-MATD3-SR (bias)"],
    ["steelblue", "darkorange", "green"],
):
    pf = extract_policy_profiles(actor_obj, env_test, init_te, n_steps=20)
    means = pf[CFG.ACTION_NAMES].mean()
    stds  = pf[CFG.ACTION_NAMES].std()

    ax.bar(
        range(len(CFG.ACTION_NAMES)),
        means.values,
        yerr=stds.values,
        capsize=4,
        color=col,
        alpha=0.75,
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_xticks(range(len(CFG.ACTION_NAMES)))
    ax.set_xticklabels(
        [n.split("_")[0] for n in CFG.ACTION_NAMES],
        rotation=45,
        ha="right",
        fontsize=8,
    )
    ax.set_title(label, fontsize=10)
    ax.set_ylabel("Среднее значение")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.4, axis="y")

plt.tight_layout()
AM.save_figure(fig, "09_policy_comparison")
print("  → output/figures/09_policy_comparison.png")
print("✅ Анализ политик завершён")


Извлечение профилей политик...
  → output/figures/08_policy_radar_clusters.png
  → output/figures/09_policy_comparison.png
✅ Анализ политик завершён


In [20]:
# ── [20] Симулированные траектории 2025-2050 по регионам ──────────────────────

def simulate_trajectory(actor, env, init_states, years, feat_cols, regions_list):
    """Симулирует многолетнюю траекторию под управлением агента."""
    # безопасно определяем длину эпизода
    ep_len = getattr(env, "ep_len", None)
    if ep_len is None:
        ep_len = getattr(env, "eplen", None)
    if ep_len is None:
        ep_len = getattr(env, "episode_len", None)
    if ep_len is None:
        ep_len = CFG.MADDPG_EPISODE_LEN

    states = env.reset(init_states.copy())
    records = []

    for yr in years:
        s_t = torch.tensor(states, dtype=torch.float32, device=DEVICE)
        actor.eval()
        with torch.no_grad():
            acts = actor(s_t).cpu().numpy()

        next_s, rws, _, info = env.step(acts)

        for ri, region in enumerate(regions_list):
            row = {
                "region": region,
                "year": yr,
                "algorithm": "policy",
                "reward": float(rws[ri]),
                "crisis": info.get("crisis"),
            }
            # сохраняем только реальные state-фичи, которые есть в состоянии
            n_feats = min(len(CFG.STATE_FEATURES), states.shape[1])
            for fi, feat in enumerate(feat_cols[:n_feats]):
                row[feat] = float(states[ri, fi])
            records.append(row)

        states = next_s

    return pd.DataFrame(records)


SIM_YEARS = list(range(2025, 2051))
print("Симуляция траекторий 2025-2050...")

# Evo-actor — bias-версия, как в ячейке 15/16
traj_evo = simulate_trajectory(
    evo_actor, env_test, init_te, SIM_YEARS, ALL_STATE_COLS, regions
)
traj_evo["algorithm"] = "Evo-MATD3-SR"

traj_maddpg = simulate_trajectory(
    maddpg_actor, env_test, init_te, SIM_YEARS, ALL_STATE_COLS, regions
)
traj_maddpg["algorithm"] = "MADDPG"

df_traj_all = pd.concat([traj_evo, traj_maddpg], ignore_index=True)
AM.save_table(df_traj_all, "09_simulated_trajectories")

# Сравнение управляемой и инерционной траектории
FEATS_PLOT = ["population", "life_expectancy", "migration_balance_rate"]
SHOWCASE2 = [
    r for r in ["Курская область", "Республика Дагестан", "Новосибирская область"]
    if r in regions
]

fig, axes = plt.subplots(len(FEATS_PLOT), 1, figsize=(14, 9))
fig.suptitle(
    "Управляемые траектории (Evo-MATD3-SR) vs инерционный прогноз",
    fontsize=12,
    fontweight="bold",
)
pal3 = sns.color_palette("husl", len(SHOWCASE2))

for ai, feat in enumerate(FEATS_PLOT):
    ax = axes[ai]
    for ri, region in enumerate(SHOWCASE2):
        # Управляемая траектория
        rt = traj_evo[traj_evo["region"] == region]
        if feat in rt.columns:
            ax.plot(
                rt["year"],
                rt[feat],
                color=pal3[ri],
                lw=2.0,
                label=f"{region} (управление)" if ai == 0 else None,
            )
        # Инерционная траектория
        rb = df_baseline[df_baseline["region"] == region]
        if feat in rb.columns:
            ax.plot(
                rb["year"],
                rb[feat],
                color=pal3[ri],
                lw=1.2,
                ls="--",
                alpha=0.6,
                label=f"{region} (baseline)" if ai == 0 else None,
            )

    ax.set_ylabel(feat.replace("_", " "), fontsize=8)
    ax.grid(True, alpha=0.4)
    ax.tick_params(labelsize=8)

axes[0].legend(fontsize=7, loc="upper right", ncol=2)
axes[-1].set_xlabel("Год")
plt.tight_layout()
AM.save_figure(fig, "10_managed_vs_baseline")
print("  → output/figures/10_managed_vs_baseline.png")
print("✅ Симуляция траекторий завершена")

Симуляция траекторий 2025-2050...
  → output/figures/10_managed_vs_baseline.png
✅ Симуляция траекторий завершена


In [21]:
# ── [21] Финальный dashboard ключевых метрик ──────────────────────────────────

def compute_demo_metrics(traj_df, feat="population"):
    """Средние агрегированные метрики по всем регионам."""
    agg = traj_df.groupby("year")[feat].mean()
    if len(agg) < 2:
        return {"trend": 0.0, "final": float(agg.iloc[-1])}
    x = np.arange(len(agg))
    slope, intercept, _, _, _ = stats.linregress(x, agg.values)
    return {
        "trend": float(slope),
        "final": float(agg.iloc[-1]),
        "series": agg,
    }


fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(3, 4, figure=fig)
fig.suptitle(
    "Финальный Dashboard: MARL-управление демографией и миграцией РФ 2025-2050",
    fontsize=13,
    fontweight="bold",
)

# ── [0,0] Сравнение наград ────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])

# results уже собран в ячейке 16: Baseline, MADDPG, MATD3, Evo-MATD3-SR
categories = [r["label"] for r in results]
vals = [r["mean_rew"] for r in results]
c0 = ["#aaaaaa", "#4c72b0", "#dd8452", "#55a868"]

ax0.bar(categories, vals, color=c0, alpha=0.85, edgecolor="black")
ax0.set_title("Episodic Reward (test)", fontsize=9)
ax0.set_ylabel("Reward")
ax0.tick_params(axis="x", labelsize=8)
ax0.grid(True, alpha=0.4, axis="y")

# ── [0,1] Динамика обучения Evo ────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
# evo_bias_log — лог из ячейки 15 (Evo-MATD3-SR bias)
df_ev = evo_bias_log if isinstance(evo_bias_log, pd.DataFrame) else pd.DataFrame()
if not df_ev.empty and "score_mean" in df_ev.columns:
    sm = pd.Series(df_ev["score_mean"].values).rolling(5, min_periods=1).mean().values
    ax1.plot(df_ev["iter"], df_ev["score_mean"], alpha=0.25, color="green", label="mean reward")
    ax1.plot(df_ev["iter"], sm, color="green", lw=2, label="MA(5)")
ax1.set_title("Evo-MATD3-SR (bias): offline эволюция", fontsize=9)
ax1.set_xlabel("Итерация Evo")
ax1.set_ylabel("Train reward (mean)")
ax1.grid(True, alpha=0.4)
ax1.legend(fontsize=7)

# ── [0,2] World Model MAE ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
wm_top = wm_mae_df.sort_values("MAE").head(8)
ax2.barh(wm_top["feature"], wm_top["MAE"], color="steelblue", alpha=0.8)
ax2.set_title("World Model MAE (top-8 feat)", fontsize=9)
ax2.tick_params(labelsize=7)
ax2.grid(True, alpha=0.4, axis="x")

# ── [0,3] Кластеры PCA ────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 3])
for cl in range(N_CLUSTERS):
    mask = cluster_labels == cl
    ax3.scatter(
        reg_pca[mask, 0],
        reg_pca[mask, 1],
        c=[pal6[cl]],
        s=20,
        alpha=0.7,
        label=f"К{cl+1}",
    )
ax3.set_title("Кластеры регионов (PCA)", fontsize=9)
ax3.legend(fontsize=6, ncol=2)
ax3.tick_params(labelsize=7)

# ── [1,:2] Средняя популяция/ОПЖ: Evo vs baseline ────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
for feat_p, col_p, lw_p in [
    ("population", "navy", 2.0),
    ("life_expectancy", "darkgreen", 1.5),
]:
    m_e = compute_demo_metrics(traj_evo, feat_p)
    m_b = compute_demo_metrics(df_baseline, feat_p)
    if "series" in m_e:
        ax4.plot(
            m_e["series"].index,
            m_e["series"],
            c=col_p,
            lw=lw_p,
            label=f"{feat_p} (Evo)",
        )
    if "series" in m_b:
        ax4.plot(
            m_b["series"].index,
            m_b["series"],
            c=col_p,
            lw=lw_p,
            ls="--",
            alpha=0.5,
            label=f"{feat_p} (baseline)",
        )

ax4.set_title("Динамика населения/ОПЖ (среднее по регионам)", fontsize=9)
ax4.legend(fontsize=7)
ax4.grid(True, alpha=0.4)
ax4.set_xlabel("Год")

# ── [1,2:] Стресс-тест Evo ────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2:])
if "Evo-MATD3-SR" in stress_results:
    dfs = stress_results["Evo-MATD3-SR"]
    ax5.bar(
        range(len(dfs)),
        dfs["mean_rew"],
        yerr=dfs["std_rew"],
        capsize=4,
        color="green",
        alpha=0.8,
    )
    ax5.set_xticks(range(len(dfs)))
    ax5.set_xticklabels(dfs["crisis"], rotation=30, ha="right", fontsize=7)
    ax5.set_title("Evo-MATD3-SR (bias): стресс-тест по кризисам", fontsize=9)
    ax5.grid(True, alpha=0.4, axis="y")
else:
    ax5.set_visible(False)

# ── [2,:] Средние политики Evo ────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, :])
pf_mean = policy_df[CFG.ACTION_NAMES].mean()
x = np.arange(len(CFG.ACTION_NAMES))
ax6.bar(x, pf_mean.values, color="seagreen", alpha=0.8, edgecolor="black", lw=0.5)
ax6.set_xticks(x)
ax6.set_xticklabels(CFG.ACTION_NAMES, rotation=30, ha="right", fontsize=9)
ax6.set_title("Evo-MATD3-SR (bias): средний профиль политик (все регионы)", fontsize=9)
ax6.set_ylabel("Среднее значение действия")
ax6.set_ylim(0, 1)
ax6.grid(True, alpha=0.4, axis="y")

plt.tight_layout(rect=[0, 0, 1, 0.96])
AM.save_figure(fig, "11_final_dashboard")
print("  → output/figures/11_final_dashboard.png")
print("✅ Dashboard сохранён")

  → output/figures/11_final_dashboard.png
✅ Dashboard сохранён


In [22]:
# ── [22] Сохранение всех артефактов и итоговый отчёт ─────────────────────────

def to_serializable(obj):
    if isinstance(obj, dict):
        return {str(k): to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def safe_model_param_count(*candidate_names):
    g = globals()
    for name in candidate_names:
        model = g.get(name, None)
        if model is not None and hasattr(model, "parameters"):
            try:
                return int(sum(p.numel() for p in model.parameters())), name
            except Exception:
                pass
    return None, None


def safe_value(name, default=None):
    return globals().get(name, default)


# World Model параметров
wm_param_count, wm_name = safe_model_param_count(
    "wm",
    "world_model",
    "wm_model",
    "world_model_ensemble",
)

# Берём финальные метрики из results (из ячейки 16)
baseline_mean = np.nan
baseline_std  = np.nan
maddpg_mean   = np.nan
maddpg_std    = np.nan
matd3_mean    = np.nan
matd3_std     = np.nan
evo_mean      = np.nan
evo_std       = np.nan

try:
    for r in results:
        if r["label"].lower().startswith("baseline"):
            baseline_mean = r["mean_rew"]
            baseline_std  = r["std_rew"]
        elif r["label"].upper() == "MADDPG":
            maddpg_mean = r["mean_rew"]
            maddpg_std  = r["std_rew"]
        elif r["label"].upper() == "MATD3":
            matd3_mean = r["mean_rew"]
            matd3_std  = r["std_rew"]
        elif "Evo-MATD3-SR" in r["label"]:
            evo_mean = r["mean_rew"]
            evo_std  = r["std_rew"]
except Exception:
    pass

print("=" * 65)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА")
print("=" * 65)
print(f"  Регионы в модели:           {N_REGIONS}")
print(f"  Признаков состояния:        {len(FEAT_COLS)}")
print(f"  Управляющих политик:        {CFG.ACTION_DIM}")

if wm_param_count is not None:
    print(f"  World Model параметров:     {wm_param_count:,}  (объект: {wm_name})")
else:
    print("  World Model параметров:     недоступно (модель не найдена в памяти)")

print("  Горизонт прогноза:          2025–2050")
print(f"  Разбивка: train ≤{CFG.YEAR_TRAIN_END} | val ≤{CFG.YEAR_VAL_END} | test {CFG.YEAR_VAL_END + 1}+")
print()

print("Результаты сравнения алгоритмов (test-среда):")
print(f"  Baseline (no action)     : {baseline_mean:.4f} ± {baseline_std:.4f}")
print(f"  MADDPG                   : {maddpg_mean:.4f} ± {maddpg_std:.4f}")
print(f"  MATD3                    : {matd3_mean:.4f} ± {matd3_std:.4f}")
print(f"  Evo-MATD3-SR (bias)      : {evo_mean:.4f} ± {evo_std:.4f}")
print()

print("Файлы в output/")
print("  tables/     → 01_data_audit_missing … 09_simulated_trajectories")
print("  figures/    → 01_region_clusters … 11_final_dashboard")
print("  checkpoints/ → world_model.pt, maddpg_actor_best.pt")
print("                matd3_actor_best.pt, evomatd3sr_bias_actor.pt")
print("  logs/       → MADDPG_log.json, MATD3_log.json, EvoMATD3SR_bias_log.json")
print("=" * 65)

summary = {
    "n_regions": int(N_REGIONS),
    "state_dim": int(len(FEAT_COLS)),
    "action_dim": int(CFG.ACTION_DIM),
    "world_model": {
        "object_name": wm_name,
        "param_count": wm_param_count,
    },
    "forecast_horizon": [2025, 2050],
    "split": {
        "train_end": int(CFG.YEAR_TRAIN_END),
        "val_end": int(CFG.YEAR_VAL_END),
        "test_start": int(CFG.YEAR_VAL_END + 1),
    },
    "results": {
        "baseline": {
            "mean": baseline_mean,
            "std": baseline_std,
        },
        "maddpg": {
            "mean": maddpg_mean,
            "std": maddpg_std,
        },
        "matd3": {
            "mean": matd3_mean,
            "std": matd3_std,
        },
        "evo_matd3_sr_bias": {
            "mean": evo_mean,
            "std": evo_std,
        },
    },
    "artifacts": {
        "tables_dir": "output/tables",
        "figures_dir": "output/figures",
        "checkpoints_dir": "output/checkpoints",
        "logs_dir": "output/logs",
    },
}

summary_safe = to_serializable(summary)

with open("output/experiment_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_safe, f, indent=2, ensure_ascii=False)

print("  → output/experiment_summary.json  ✅")

ИТОГОВЫЕ РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА
  Регионы в модели:           82
  Признаков состояния:        10
  Управляющих политик:        9
  World Model параметров:     543,036  (объект: world_model)
  Горизонт прогноза:          2025–2050
  Разбивка: train ≤2012 | val ≤2018 | test 2019+

Результаты сравнения алгоритмов (test-среда):
  Baseline (no action)     : -0.3044 ± 0.0407
  MADDPG                   : 2.0044 ± 0.0311
  MATD3                    : 2.2022 ± 0.0189
  Evo-MATD3-SR (bias)      : 2.2331 ± 0.0063

Файлы в output/
  tables/     → 01_data_audit_missing … 09_simulated_trajectories
  figures/    → 01_region_clusters … 11_final_dashboard
  checkpoints/ → world_model.pt, maddpg_actor_best.pt
                matd3_actor_best.pt, evomatd3sr_bias_actor.pt
  logs/       → MADDPG_log.json, MATD3_log.json, EvoMATD3SR_bias_log.json
  → output/experiment_summary.json  ✅


In [23]:
# ── [22.1] Архивация и скачивание всех данных эксперимента ───────────────────

import os
import zipfile
from pathlib import Path

OUTPUT_ROOT = Path("output")
ARCHIVE_PATH = Path("output_experiment_bundle.zip")

if not OUTPUT_ROOT.exists():
    raise FileNotFoundError("Папка output не найдена. Сначала выполните ячейки сохранения артефактов.")

# удаляем старый архив, если уже был
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()

file_count = 0
total_size = 0

with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_ROOT.rglob("*"):
        if path.is_file():
            # на всякий случай не включаем сам архив, если он вдруг лежит внутри output
            if path.resolve() == ARCHIVE_PATH.resolve():
                continue
            arcname = path.relative_to(Path("."))
            zf.write(path, arcname=arcname)
            file_count += 1
            total_size += path.stat().st_size

archive_size_mb = ARCHIVE_PATH.stat().st_size / (1024 ** 2)
total_size_mb = total_size / (1024 ** 2)

print("✅ Архивация завершена")
print(f"  Файлов в архиве:         {file_count}")
print(f"  Суммарный размер файлов: {total_size_mb:.2f} MB")
print(f"  Архив:                   {ARCHIVE_PATH}")
print(f"  Размер архива:           {archive_size_mb:.2f} MB")
print("  В архив включены таблицы, фигуры, логи и checkpoint'ы, включая evomatd3sr_bias_actor.pt")

# попытка показать ссылку на скачивание в Jupyter / Colab
try:
    from IPython.display import FileLink, display
    display(FileLink(str(ARCHIVE_PATH)))
    print("  Ссылка на скачивание создана")
except Exception as e:
    print(f"  Не удалось создать ссылку на скачивание: {e}")

# если это Google Colab — можно сразу скачать
try:
    from google.colab import files
    print("  Для прямого скачивания раскомментируйте следующую строку:")
    print(f"  files.download('{ARCHIVE_PATH.name}')")
except Exception:
    pass

✅ Архивация завершена
  Файлов в архиве:         27
  Суммарный размер файлов: 6.12 MB
  Архив:                   output_experiment_bundle.zip
  Размер архива:           4.66 MB
  В архив включены таблицы, фигуры, логи и checkpoint'ы, включая evomatd3sr_bias_actor.pt


/content/output_experiment_bundle.zip

  Ссылка на скачивание создана
  Для прямого скачивания раскомментируйте следующую строку:
  files.download('output_experiment_bundle.zip')


---
## 📦 Краткая схема пайплайна

| Ячейка | Назначение |
|--------|-----------|
| 1–4    | Установка, импорты, конфигурация, утилиты |
| 5–6    | Загрузка данных, нормализация, разбивка |
| 7      | Кластеризация + индекс уязвимости |
| 8      | Инерционный прогноз ETS до 2050 |
| 9–11   | Ансамблевая GRU World Model — архитектура, обучение, валидация |
| 12     | MARL-среда `RussiaRegionsEnv` + кризисный модуль |
| 13     | MADDPG (shared actor/critic, replay buffer) |
| 14     | MATD3 (twin critic, policy delay, target noise) |
| 15     | Evo-MATD3-SR (MetaMonitor + MetaEditor + PolicyArchive) |
| 16     | Сравнительная оценка на test-среде |
| 17     | Кривые обучения |
| 18     | Стресс-тест по 6 кризисным сценариям |
| 19     | Анализ политик: радары по кластерам |
| 20     | Симулированные траектории 2025–2050 |
| 21     | Финальный dashboard |
| 22     | Сохранение всех артефактов |
